# Goodreads Machine Learning - Group Project
------------
------------


## 1. Data loading and first look


#### Load project libraries

In [ ]:
import re
import unicodedata
from typing import Any

import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


pd.set_option("display.max_colwidth", None)

### Load source files


In [ ]:
CSV_RAW_PATH = "../data/raw/books.csv"
df_raw_parse_check = pd.read_csv(
    CSV_RAW_PATH, index_col="bookID", on_bad_lines='warn')

Upon inspection, the 4 errors came from the fact that the author string contained a comma "," which was interpreted as a new column and spilled the author name into the next column when creating the .csv, thus resulting in 13 columns instead of 12. This file was manually repaired by removing the commas in the author column.


#### Load the repaired dataset

In [ ]:
CSV_REPAIRED_PATH = "../data/processed/books-hugo.csv"
df_raw = pd.read_csv(CSV_REPAIRED_PATH, on_bad_lines="warn")


We will keep 3 data frames through the notebook :
- `df_raw` for the loaded repaired file
- `df_clean` for the cleaned version of the source data (later used for actual modelling)
- `df_model` for modelling-oriented EDA, engineered features, and row filters


### First look


In [ ]:
df_raw.head()


In [ ]:
df_raw.info()


We can see some issue with leading space in the column names, as well as some missing values in the `publication_date` column being represented as strings.

------------
## 2. Cleaning / EDA


### 2.1 General cleaning


#### Initial observations
- It looks like there is a leading space for the " num_pages" column name
- isbn13 as int64 seems like the wrong type for an identifier, but not a big deal because we will probably drop these columns
- publication_date as string is not ideal, should be parsed to date


#### Check identifier uniqueness

Confirming whether the book identifiers are unique before choosing one of them as the dataframe index.


In [ ]:
# check if bookID, ISBN, ISBN13 are all unique
identifier_columns = ["bookID", "isbn", "isbn13"]
{column_name: df_raw[column_name].is_unique for column_name in identifier_columns}


We set `bookID` as the index because it is unique and easier to use for row-level corrections later.

In [ ]:
df_raw.set_index("bookID", inplace=True)

#### Trim leading and trailing spaces


In [ ]:
# fix column names
df_raw.columns = df_raw.columns.str.strip()

#### Check string whitespace

Scanning text columns for leading or trailing spaces before copying the data into the cleaning dataframe.


In [ ]:
# test if any of the values contains leading/trailing spaces
string_columns = df_raw.select_dtypes(include=["object", "string"]).columns
for column_name in string_columns:
    has_leading_or_trailing_spaces = (
        df_raw[column_name]
        .dropna()
        .ne(df_raw[column_name].dropna().str.strip())
        .any()
    )
    if has_leading_or_trailing_spaces:
        print(f"!! Leading/trailing spaces detected in column: {column_name}")
    else:
        print(f"no spaces detected on column : {column_name}")


#### Create the cleaning dataframe

We keep `df_raw` unchanged and do all cleaning work on `df_clean`.


In [ ]:
df_clean = df_raw.copy()


#### Trim title values

We remove surrounding spaces from titles before applying broader text normalization.


In [ ]:
# fix spaces in titles
df_clean["title"] = df_clean["title"].str.strip()


#### Normalize strings
Define text normalization helper

In [ ]:
def normalize_string(
    value: Any,
    *,
    lowercase: bool = True,
    strip_accents: bool = True,
    collapse_whitespace: bool = True,
    remove_suffixes: bool = False,
    remove_punctuation: bool = False,
) -> str | None:
    """
    Normalize a single string for fuzzy matching / grouping keys.
    Returns None for None/NaN. Non-strings are cast to str first.
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    normalized_text = str(value).strip()
    if not normalized_text:
        return ""
    if lowercase:
        normalized_text = normalized_text.lower()
    if strip_accents:
        normalized_text = unicodedata.normalize("NFKD", normalized_text)
        normalized_text = "".join(
            character for character in normalized_text if not unicodedata.combining(character)
        )
    if collapse_whitespace:
        normalized_text = re.sub(r"\s+", " ", normalized_text)
    if remove_suffixes:
        normalized_text = re.sub(
            r"\s+(jr\.?|sr\.?|iii|ii|iv)$",
            "",
            normalized_text,
            flags=re.IGNORECASE,
        ).strip()
    if remove_punctuation:
        normalized_text = re.sub(r"[^\w\s]", "", normalized_text)
        normalized_text = re.sub(r"\s+", " ", normalized_text).strip()
    return normalized_text


#### Normalize main text fields

In [ ]:
df_clean["title"] = df_clean["title"].apply(normalize_string)
df_clean["authors"] = df_clean["authors"].apply(normalize_string)
df_clean["publisher"] = df_clean["publisher"].apply(normalize_string)


#### Blank, empty, null, and bad data


In [ ]:
df_clean.describe()

At first glance, we can see that there are rows with num_pages and text_reviews_count equal to 0  
➜ should be investigated and potentially cleaned


#### Check explicit missing values

In [ ]:
df_clean.isna().sum()


#### Check null-like text values

In [ ]:
# check for bad data
null_like_values = ["", "nan", "None", "N/A", "null", "-"]
null_like_value_counts = {}
for column_name in df_clean.columns:
    null_like_mask = df_clean[column_name].astype(str).str.strip().str.lower().isin(null_like_values)
    null_like_value_counts[column_name] = int(null_like_mask.sum())
null_like_value_counts


➜ no strictly missing values
But visual analysis show that multiple entries have "NOT A BOOK" as authors


#### Count invalid author labels

In [ ]:
print("number of books with \'NOT A BOOK\' as authors :", (df_clean["authors"] == "NOT A BOOK".lower()).sum())

In [ ]:
df_clean.loc[df_clean["authors"] == "NOT A BOOK".lower(),:]

This looks like audiobooks or podcasts
**[Decision]** Let's drop them, even if they are audiobooks and have a rating, they should have an author


#### Drop invalid author rows

In [ ]:
df_clean = df_clean[df_clean["authors"] != "NOT A BOOK".lower()]

#### Zero values


In [ ]:
print("num_pages zero count :", (df_clean["num_pages"] == 0).sum())
print("ratings_count zero count :", (df_clean["ratings_count"] == 0).sum())
print("text_reviews_count zero count :", (df_clean["text_reviews_count"] == 0).sum())

➜ These should be investigated and potentially cleaned / dropped (see following sections)


### 2.2 `publication_date`


#### Convert publication date to datetime


In [ ]:
# fix publication_date, passing format to parse
raw_publication_date = df_clean["publication_date"].copy()
df_clean["publication_date"] = pd.to_datetime(
    df_clean["publication_date"],
    format="%m/%d/%Y",
    errors="coerce", # if the format is not correct, set to NaT
)
# check original values where the conversion failed
raw_publication_date.loc[df_clean["publication_date"].isna()]


2 dates did not convert properly : these dates are impossible, let's check ISBN code and look for the information on the internet since there are only 2 books with invalid dates


#### Inspect invalid publication dates

In [ ]:
df_clean.loc[df_clean.publication_date.isna(), ["isbn", "isbn13", "title"]]

According to current goodreads.com: 
- *In Pursuit of the Proper Sinner* was published in October 31st, 2000
- *Montaillou  village occitan de 1294 à 1324* was published in June 30, 1982

One date is wrong by 1 day, the other by 1 month. 

These errors could be isolated, or might be a symptom of a larger issue with dates. Some of the parsable date might still be wrong. Goodreads.com does not provide a public API to check the data programmatically (an unofficial one exists on Apify), but there are other options to check the publishing date with Open Library or Google Books API.  

Let's apply a target fix:


**[Decision]** Since there are only 2 impossible dates, let's correct them manually and keep the rows.


#### Correct the two bad dates

Manually fixing the two publication dates already checked against Goodreads because the errors are isolated.


In [ ]:
# fix the dates by index
df_clean.loc[31373, "publication_date"] = pd.to_datetime("2000-10-31")
df_clean.loc[45531, "publication_date"] = pd.to_datetime("1982-06-30")

#### Initialize modelling dataframe and publication year

Starting `df_model` from the cleaned data before adding modelling features. Here we add the publication year as a feature as it is easier to use than an exact date in EDA and modelling.


In [ ]:
df_model = df_clean.copy()
df_model["publication_year"] = df_model["publication_date"].dt.year

### 2.3 `average_rating`


#### Plot average rating distribution

In [ ]:
average_ratings = df_model["average_rating"].dropna()
mean_rating = average_ratings.mean()

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(average_ratings, bins=40, kde=True, color="#2a9d8f", ax=ax)
ax.axvline(
    mean_rating,
    color="#e76f51",
    linestyle="--",
    linewidth=2,
    label=f"Mean = {mean_rating:.3f}",
)
ax.set(
    xlim=(0, average_ratings.max()),
    title="Average Rating - Distribution Around the Mean",
    xlabel="Average Rating",
    ylabel="Count",
)
ax.legend()
plt.tight_layout()
plt.show()

average_rating_distribution_metrics = pd.DataFrame({
    "metric": ["mean", "standard_deviation", "minimum", "maximum", "count"],
    "value": [
        mean_rating,
        average_ratings.std(),
        average_ratings.min(),
        average_ratings.max(),
        average_ratings.count(),
    ],
})

average_rating_distribution_metrics


#### Interpretation
- distribution is heavily centered around the average: 3.93
- a few books appear with average rating equal to zero (no ratings?)


#### Check impossible ratings

We verify that there are no negative average ratings before creating rating bands.


In [ ]:
print("average_rating negative count :",(df_model["average_rating"] < 0).sum())


#### Create average-rating deciles

We split the target into ten groups to compare the spread of low, middle, and high rating ranges.


In [ ]:
average_rating_decile_labels = [
    "N1", "N2", "N3", "N4", "N5", "N6", "N7", "N8", "N9", "N10"
]
df_model["average_rating_decile"] = pd.qcut(
    df_model["average_rating"],
    q=10,
    labels=average_rating_decile_labels,
)
df_model.groupby("average_rating_decile")["average_rating"].describe()


Standard deviation is worse for the lowest (N1) and highest (N10) average rating groups.


### 2.4 `ratings_count`


#### First Look


In [ ]:
df_model["ratings_count"].describe()


#### Relation with `average_rating`


In [ ]:
# Plot how average_rating changes as ratings_count increases.
sns.regplot(x="average_rating", y="ratings_count", data=df_model)
plt.show()


#### Use log rating counts for readability


In [ ]:
# Keep only rows with a positive ratings count
ratings_count_correlation_df = df_model.loc[
    df_model["ratings_count"] > 0,
    ["ratings_count", "average_rating"],
].copy()

ratings_count_plot_df = ratings_count_correlation_df.copy()
ratings_count_plot_df["log_ratings_count"] = np.log10(ratings_count_plot_df["ratings_count"])

# Plot average_rating against log-scaled ratings_count to make dense low-count books easier to compare.
plt.figure(figsize=(8, 5))
sns.regplot(data=ratings_count_plot_df, x="log_ratings_count", y="average_rating",
            scatter_kws={"alpha": 0.2}, line_kws={"color": "red"})
plt.title("Average rating vs ratings_count")
plt.xlabel("log10(ratings_count)")
plt.ylabel("average_rating")
plt.tight_layout()
plt.show()


We can see in this graph that all 0-2 stars and all 5 stars books have less than 10 ratings, but there is not a clear correlation between ratings_count and average_rating. It also shows that there are a few books with a positive average rating but ratings_count = 0. This is an anomaly and should be investigated.


#### Ratings_count anomalies

We separate zero-rating rows with positive ratings from rows with both zero ratings and zero average rating.


In [ ]:
# number of books with ratings_count = 0 and average_rating > 0
print(f"Number of books with ratings_count = 0 and average_rating > 0: {len(df_model[(df_model['ratings_count'] == 0) & (df_model['average_rating'] > 0)])}")
# number of books with ratings_count = 0 and average_rating = 0
print(f"Number of books with ratings_count = 0 and average_rating = 0: {len(df_model[(df_model['ratings_count'] == 0) & (df_model['average_rating'] == 0)])}")


#### Build rating-count bands

We group books by rating support so the mean rating and rating spread can be compared by band.


In [ ]:
ratings_count_bins = [-1, 0, 9, 25, 100, 1_000, 10_000, np.inf]
ratings_count_labels = ["0", "1-9", "10-25",
                        "26-100", "101-1k", "1k-10k", "10k+"]

df_model["ratings_count_band"] = pd.cut(
    df_model["ratings_count"],
    bins=ratings_count_bins,
    labels=ratings_count_labels,
    include_lowest=True,
)

ratings_count_summary = (
    df_model.groupby("ratings_count_band", observed=True)
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
        rating_std=("average_rating", "std"),
        median_ratings_count=("ratings_count", "median"),
    )
    .reset_index()
)
ratings_count_summary["share_of_df_model"] = ratings_count_summary["books"] / len(df_model)

ratings_count_summary

#### Plot rating-count band summary

In [ ]:
# Compare the number of books in each rating-count band with the rating spread inside each band.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(
    data=ratings_count_summary,
    x="ratings_count_band",
    y="books",
    color="#2a9d8f",
    ax=axes[0],
)
axes[0].set_title("Books by rating-support band")
axes[0].set_xlabel("Ratings count band")
axes[0].set_ylabel("Number of books")
axes[0].tick_params(axis="x", rotation=30)

sns.lineplot(
    data=ratings_count_summary,
    x="ratings_count_band",
    y="rating_std",
    marker="o",
    color="#e76f51",
    ax=axes[1],
)
axes[1].set_title("Rating spread is wider for low-rating_count books")
axes[1].set_xlabel("Ratings count band")
axes[1].set_ylabel("Std. dev. of average_rating")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


The average rating for each rating_count band is pretty close to the average rating of the dataset. One hypothesis could be that the best books would have more engagements (more ratings), but on average this is not the case for this dataset. The 10k+ band does show a slightly higher average rating, but the difference is not very large.


**[Decision]** Rows with `ratings_count = 0` should not be used for modelling, because even if they have a rating, the data must be corrupted. Furthermore, as shown in the graph, 0-2 stars and 5 stars books are outliers, allowed only because of the low number of ratings. Standard deviation quickly stabilizes when there are more than 10 ratings, it is therefore reasonable to drop books with less than 10 ratings.


#### Filtering df_model by dropping rows with ratings_count < 10


This threshold is arbitrary for now but will be optimized during the modelling phase.

In [ ]:
# drop rows with ratings_count < 10 from the modelling dataframe
df_model = df_model[df_model["ratings_count"] > 9].copy()


#### Check rows kept for modelling

In [ ]:
print('rows in df_clean :', len(df_clean))
print('rows in df_model :', len(df_model))
#numer of removed rows
print('number of removed rows :', len(df_clean) - len(df_model))

876 is a lot of rows to remove (~8% of the dataset), but it is necessary to avoid the model relying on bad data.

#### Recheck rating-count relationship

In [ ]:
# Plot how average_rating changes as ratings_count increases after feature cleaning.
sns.regplot(x="average_rating", y="ratings_count", data=df_model)
plt.show()

### 2.5 `num_pages`


#### First look

In [ ]:
# describe num_pages
df_model["num_pages"].describe()


#### Plot num_pages distribution

In [ ]:
# Plot the distribution of page counts to show where most books are concentrated.
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df_model["num_pages"], bins=100, ax=ax)
ax.set_title("Distribution of number of pages")
ax.set_xlabel("Number of pages")
ax.set_ylabel("Count")
plt.show()


#### Count unusual number of pages


In [ ]:
LOW_PAGE_COUNT_THRESHOLD = 10
HIGH_PAGE_COUNT_THRESHOLD = 2000

print(
    f'books with less than {LOW_PAGE_COUNT_THRESHOLD} pages : {(df_model["num_pages"]<LOW_PAGE_COUNT_THRESHOLD).sum()}\n'
    f'books with more than {HIGH_PAGE_COUNT_THRESHOLD} pages : {(df_model["num_pages"]>HIGH_PAGE_COUNT_THRESHOLD).sum()}\n'
    f'books with zero pages : {(df_model["num_pages"] == 0).sum()}'
)
 


#### Inspect zero-page books

In [ ]:
# show first 20 books with zero pages
df_model[df_model["num_pages"] == 0][["title", "authors", "num_pages", "average_rating", "publisher"]].head(20)

It looks like most of these books are audiobooks (as suggested by the name of the publishers), explaining why the page count is not available.


#### Check zero-page rating level

In [ ]:
# avegare rating for books with zero pages
print(f"average rating for books with zero pages : {round(df_model[df_model['num_pages'] == 0]['average_rating'].mean(), 2)}")


#### Review zero-page publishers

In [ ]:
# check distinct publishers for books with zero pages
df_model[df_model['num_pages'] == 0]["publisher"].value_counts()


#### Identifying audiobooks


Defining audiobook patterns

In [ ]:
# create a regex pattern to identify audiobooks
AUDIO_REGEX_FLAGS = re.IGNORECASE | re.VERBOSE

AUDIO_INDICATOR_PATTERN = r"""
audio(?:books?|go|text)?\b
|
\bcassettes?\b
|
\bcd\s*audio\b
|
\baudio\s*cd\b
|
\bmp3\b
|
\bsound\s+recording\b
|
\blistening\s+library\b
"""


#### Flag likely audiobooks

Searching title and publisher text for audio markers and store the result in `is_audio`.


In [ ]:
title_publisher_text = (
    df_model["title"].fillna("") + " " + df_model["publisher"].fillna("")
)

df_model["is_audio"] = title_publisher_text.str.contains(
    AUDIO_INDICATOR_PATTERN, regex=True, flags=AUDIO_REGEX_FLAGS, na=False
)

#### Summarize audiobook rows

In [ ]:
# count the number of audiobooks
print(f"Number of audiobooks : {df_model['is_audio'].sum()}")

# average/median number of pages for audiobooks with a page count > 0
print(
    f"Average number of pages for audiobooks with a page count > 0 : {round(df_model[(df_model['is_audio'] == 1) & (df_model['num_pages'] > 0)]['num_pages'].mean(), 1)}")
print(
    f"Median number of pages for audiobooks with a page count > 0 : {round(df_model[(df_model['is_audio'] == 1) & (df_model['num_pages'] > 0)]['num_pages'].median(), 1)}")

# average/median rating for audiobooks
print(f"Average rating for audiobooks : {round(df_model[df_model['is_audio'] == 1]['average_rating'].mean(), 2)}")
print( f"Median rating for audiobooks : {round(df_model[df_model['is_audio'] == 1]['average_rating'].median(), 2)}")

# show first 20 books ordered by num_pages
df_model[df_model["is_audio"] == 1][["title", "authors", "num_pages", "average_rating", "publisher"]].sort_values(by="num_pages", ascending=False).head(20)


So even though some audiobooks have zero pages, it is not always the case. One even have more than 1000 pages, and the average is about 33 pages (for audiobooks with num_pages > 0).


#### Comparing low-page audio status

In [ ]:
# Check how many books with a page count less than LOW_PAGE_COUNT_THRESHOLD are audiobooks vs non-audiobooks
low_page_count_books = df_model[df_model['num_pages'] < LOW_PAGE_COUNT_THRESHOLD].shape[0]
low_page_count_audio_books = df_model[(df_model['num_pages'] < LOW_PAGE_COUNT_THRESHOLD) & (df_model['is_audio'] == 1)].shape[0]
low_page_count_non_audio_books = df_model[(df_model['num_pages'] < LOW_PAGE_COUNT_THRESHOLD) & (df_model['is_audio'] == 0)].shape[0]

print(f"Total number of books with num_pages < {LOW_PAGE_COUNT_THRESHOLD}: {low_page_count_books}")
print(f"Number of audiobooks with num_pages < {LOW_PAGE_COUNT_THRESHOLD}: {low_page_count_audio_books}")
print(f"Number of non-audiobooks with num_pages < {LOW_PAGE_COUNT_THRESHOLD}: {low_page_count_non_audio_books}")


On the other hand, although the publisher most often contains the word "audio" or similar, it is not always the case for books with zero pages, so we need to investigate these further.


#### Inspecting non-audio zero-page rows

In [ ]:
# Show first 20 books with num_pages = 0 and is_audio = 0
df_model[(df_model["num_pages"] == 0) & (df_model["is_audio"] == 0)][["title", "authors", "num_pages", "average_rating","ratings_count", "publisher"]].head(20)


#### Selecting a publisher to investigate further

In [ ]:
# show first 10 books with a specific publisher
publisher_lookup_value = "basic books"
df_model[df_model["publisher"] == publisher_lookup_value][["title", "authors", "num_pages", "average_rating", "publisher"]].head(10)


From this result it is clear that books with zero pages are actually anomalies, and should be cleaned because :
- most audiobooks do have a page count > 0
- books with zero pages are not always audiobooks


#### Cleaning books with zero pages


**[Decision]** One way to clean the data is to keep the rows, but treat `num_pages = 0` as missing page information rather than a literal zero-page book.
- create a new column `num_pages_missing` to indicate if the page count is missing
- use the median of the valid page counts to impute the missing values (per audio/non-audio group)


Marking zero page counts as missing and fill them with audiobook or non-audiobook medians.

In [ ]:
# Flag rows where page count is missing / invalid
df_model["num_pages_missing"] = df_model["num_pages"].eq(0).astype(int)

# Create a cleaned page-count feature
df_model["num_pages_clean"] = df_model["num_pages"].replace(0, np.nan)

audio_page_count_median = round(df_model[df_model["is_audio"] == 1]["num_pages_clean"].median(), 0)
non_audio_page_count_median = round(df_model[df_model["is_audio"] == 0]["num_pages_clean"].median(), 0)

print(f"Median page count for audiobooks : {audio_page_count_median}")
print(f"Median page count for non-audiobooks : {non_audio_page_count_median}")

# Impute with the median of valid page counts for each group
df_model.loc[df_model["is_audio"] == 1, "num_pages_clean"] = df_model.loc[df_model["is_audio"] == 1, "num_pages_clean"].fillna(audio_page_count_median)
df_model.loc[df_model["is_audio"] == 0, "num_pages_clean"] = df_model.loc[df_model["is_audio"] == 0, "num_pages_clean"].fillna(non_audio_page_count_median)

# check the results for books with zero pages
df_model[df_model["num_pages_missing"] == 1][["title", "num_pages", "num_pages_clean", "publisher"]].head(10)


#### Dropping the num_pages column and renaming num_pages_clean to num_pages


In [ ]:
# drop the num_pages column
df_model = df_model.drop(columns=["num_pages"])
# rename num_pages_clean to num_pages
df_model = df_model.rename(columns={"num_pages_clean": "num_pages"})


#### Continuing investigation into high page-count books


In [ ]:
df_model.loc[df_model["num_pages"]>HIGH_PAGE_COUNT_THRESHOLD,["title","num_pages", "average_rating"]]


Most of the books with more than 2000 pages are actually sets of multiple books, but not all of them. Their average rating is also higher than the average rating of the dataset.


#### Looking at the correlation between num_pages and average_rating

In [ ]:
# Compare the cleaned raw page-count feature to average_rating
page_count_correlation_df = df_model[["num_pages", "average_rating"]].copy()

# Pearson: linear relationship
page_count_pearson_corr = page_count_correlation_df["num_pages"].corr(
    page_count_correlation_df["average_rating"], method="pearson")

# Spearman: rank/monotonic relationship, usually better here because pages are skewed
page_count_spearman_corr = page_count_correlation_df["num_pages"].corr(
    page_count_correlation_df["average_rating"], method="spearman")

print("Pearson correlation (cleaned num_pages):", round(page_count_pearson_corr, 4))
print("Spearman correlation (cleaned num_pages):", round(page_count_spearman_corr, 4))
print("Rows used:", len(page_count_correlation_df))


In [ ]:
# Plot the relationship between cleaned page counts and average_rating.
plt.figure(figsize=(8, 5))
sns.regplot(
    data=page_count_correlation_df,
    x="num_pages",
    y="average_rating",
    scatter_kws={"alpha": 0.15},
    line_kws={"color": "red"},
)
plt.title("Average rating vs cleaned num_pages")
plt.xlabel("Cleaned num_pages")
plt.ylabel("average_rating")
plt.tight_layout()
plt.show()


The cleaned raw `num_pages` feature shows only a weak or no relationship with `average_rating`. To capture broader patterns more clearly, we can group books into num_pages bands and inspect the average rating by band.


#### Creating num_pages bands

In [ ]:
page_count_bins = [0, 50, 100, 200, 300, 500, 800, 1_200, np.inf]
page_count_labels = ["1-50", "51-100", "101-200", "201-300", "301-500", "501-800", "801-1200", "1200+"]

df_model["page_count_band"] = pd.cut(
    df_model["num_pages"],
    bins=page_count_bins,
    labels=page_count_labels,
    include_lowest=True,
)

page_count_band_summary = (
    df_model.groupby("page_count_band", observed=True)
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
        rating_std=("average_rating", "std"),
        median_num_pages=("num_pages", "median"),
    )
    .reset_index()
)
page_count_band_summary["share_of_df_model"] = page_count_band_summary["books"] / len(df_model)

page_count_band_summary


In [ ]:
# Plot how mean average_rating changes across page-count bands.
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=page_count_band_summary,
    x="page_count_band",
    y="mean_rating",
    marker="o",
    color="#2a9d8f",
)
plt.title("Average rating by cleaned page-count band")
plt.xlabel("Cleaned page-count band")
plt.ylabel("Mean average_rating")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


-> to be determined whether to use page_count_band as a feature or keep num_pages, depending on the model and performance


### 2.6 `text_reviews_count`


#### First look


In [ ]:
# Describe text_reviews_count
df_model["text_reviews_count"].describe()

In [ ]:
# Choose a cutoff for text_reviews_count visualization, e.g., 99th percentile
# text_reviews_plot_cutoff = df_model["text_reviews_count"].quantile(0.99)
text_reviews_plot_cutoff = 4000
text_reviews_for_plot = df_model[df_model["text_reviews_count"] <= text_reviews_plot_cutoff]["text_reviews_count"]

# Plot the distribution of text review counts after trimming extreme outliers for readability.
plt.figure(figsize=(10, 5))
sns.histplot(text_reviews_for_plot, bins=40)
plt.title("Distribution of text_reviews_count (< 4000)")
plt.xlabel("text_reviews_count (<= {:.0f})".format(text_reviews_plot_cutoff))
plt.ylabel("Count of Books")
plt.tight_layout()
plt.show()

print(f"Number of books with text_reviews_count > 4000: {len(df_model[df_model['text_reviews_count'] > 4000])}")


More than half the books in the dataset have less than 100 reviews, and the distribution is right skewed. Using the log scale might makes the distribution more readable and easier to model for linear models.


#### Inspecting most-reviewed books

In [ ]:
# first 10 books, ordered by text_reviews_count
df_model[["title", "authors", "text_reviews_count", "average_rating", "ratings_count"]].sort_values(by="text_reviews_count", ascending=False).head(10)


#### Inspecting books with no text reviews

In [ ]:
# zero reviews count
zero_text_review_books = df_model.loc[df_model["text_reviews_count"] == 0][["title", "authors", "average_rating", "ratings_count"]]
zero_text_review_books.head(10)


In [ ]:
# average rating for books with zero reviews
print(f"average rating for books with zero reviews : {round(zero_text_review_books['average_rating'].mean(), 2)}")


The average rating for books with zero reviews is pretty much the same as the average rating for the dataset, so having no reviews is not a sign that a book is bad.


Because the spread is so large, it is useful to compare the relationship between `text_reviews_count` and `average_rating` on the raw scale and on a log scale.


#### Comparing raw and log review scales

In [ ]:
# Compare the raw and log-scaled relationship between text review counts and average_rating.
text_reviews_correlation_df = df_model[["text_reviews_count", "average_rating"]].copy()
text_reviews_correlation_df["text_reviews_count_log"] = np.log1p(text_reviews_correlation_df["text_reviews_count"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.regplot(
    data=text_reviews_correlation_df,
    x="text_reviews_count",
    y="average_rating",
    scatter_kws={"alpha": 0.12},
    line_kws={"color": "red"},
    ax=axes[0],
)
axes[0].set_title("Average rating vs text_reviews_count")
axes[0].set_xlabel("text_reviews_count")
axes[0].set_ylabel("average_rating")

sns.regplot(
    data=text_reviews_correlation_df,
    x="text_reviews_count_log",
    y="average_rating",
    scatter_kws={"alpha": 0.12},
    line_kws={"color": "red"},
    ax=axes[1],
)
axes[1].set_title("Average rating vs log1p(text_reviews_count)")
axes[1].set_xlabel("log1p(text_reviews_count)")
axes[1].set_ylabel("average_rating")

plt.tight_layout()
plt.show()


The raw and log-scaled versions both remain difficult to read because the relationship is weak and the feature is highly skewed. This feature might not matter much for the model, but it is kept for now.


### 2.7 `language_code`


#### First look


In [ ]:
language_value_counts = df_model["language_code"].value_counts(dropna=False)
language_value_counts

A lot of languages are unique or very rare, we might want to group them together for modelling.


#### Compare ratings by language

In [ ]:
# Plot mean average_rating by language_code with 95% confidence intervals.

fig, ax = plt.subplots(figsize=(9, 5))

language_value_counts = df_model["language_code"].value_counts(dropna=False)

language_rating_plot_df = df_model[["language_code", "average_rating"]].copy()

language_rating_order = (
    language_rating_plot_df.groupby("language_code", dropna=False)["average_rating"]
    .mean()
    .sort_values(ascending=False)
    .index
)

sns.pointplot(
    data=language_rating_plot_df,
    x="language_code",
    y="average_rating",
    order=language_rating_order,
    errorbar=("ci", 95),
    color="#2a9d8f",
    ax=ax,
)
ax.set_title("Language & ratings")
ax.set_xlabel("Language Code")
ax.set_ylabel("Mean average rating")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


Even though a lot of non-english books have a higher average rating, this might just be due to the fact that there are less of them in the dataset.

#### Finding multi-language titles

In [ ]:
# check for multi-language books
multi_language_books = (
    df_model.groupby(["title", "authors", "average_rating"])["language_code"]
    .agg(n_languages="nunique", languages=lambda language_codes: sorted(language_codes.dropna().unique()))
    .reset_index()
    .query("n_languages > 1")
    .sort_values("n_languages", ascending=False)
    
)
multi_language_books.head(10)

In [ ]:
# number of multi-language books
print("number of multi-language books :", len(multi_language_books))

Most books in the dataset are not multi-language (although this might be due to the title being different in different languages, but we have no way to link them if they exist in the dataset). Most books returned here are actually of the same language (english) but of different type of english.

#### Average rating for multi-language books

In [ ]:
# average rating for multi-language books
round(multi_language_books["average_rating"].mean(), 2)


The average rating for books with "multiple languages" is about the same as the average rating for the dataset -> not a strong indicator of the quality of the book. We might want to test the models without this feature first and add it as a sensitivity to see if it improves the model.


#### Modelling : Group rare languages

We keep frequent language codes and combine the rest into `other_language` to avoid a sparse categorical feature.


In [ ]:
# Create a grouped language-code feature
LANGUAGE_GROUP_MIN_COUNT = 50

language_counts_for_grouping = df_model["language_code"].value_counts(dropna=False)
common_language_codes = language_counts_for_grouping[
    language_counts_for_grouping >= LANGUAGE_GROUP_MIN_COUNT
].index

df_model["language_code_grouped"] = df_model["language_code"].where(
    df_model["language_code"].isin(common_language_codes),
    "other_language",
)

language_grouped_summary = (
    df_model.groupby("language_code_grouped")
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
    )
    .reset_index()
    .sort_values("books", ascending=False)
)
language_grouped_summary["share_of_df_model"] = language_grouped_summary["books"] / len(df_model)
print("Language codes kept as their own group:", sorted(common_language_codes))

language_grouped_summary


The grouped language feature keeps frequent language codes as separate categories and combines rare codes into `other_language`. Most well represented language have an average rating close to the average rating of the dataset, but for rare languages, the average rating is significantly higher.


### 2.8 `authors`


Visual inspection shows that some books have a lot of authors, separated by a "/". We might want to separate the strings to identify each author.


#### Count authors per book

In [ ]:
# check what is the maximum number of authors for one book
author_count_per_book = df_model["authors"].str.split("/").str.len()
print("max number of authors for one book:", author_count_per_book.max())


51 authors for one book !


In [ ]:
author_preview_columns = ["title", "authors"]
df_model.loc[author_count_per_book == author_count_per_book.max(), author_preview_columns]


**[Decision]** To avoid empty data, and because the first author is usually the main one (confirmed below), let's add a column for the first author only, and one column with the number of authors


#### Creating author features

In [ ]:
# creating new columns
df_model["first_author"] = df_model["authors"].str.split(
    "/").str[0].str.strip()
df_model["n_authors"] = author_count_per_book


#### Ploting author-count distribution

In [ ]:
# Plot the distribution of author counts to show how many authors most books have.
plt.figure(figsize=(10, 6))
sns.histplot(df_model["n_authors"], bins=range(1, author_count_per_book.max() + 2))
plt.title("Distribution of Number of Authors per Book")
plt.xlabel("Number of Authors")
plt.ylabel("Count")
plt.show()


It looks like books with more than 10 authors are very rare and the majority of books have only one author.


#### Full author list vs first author comparison

In [ ]:
# Print number of unique authors vs first author
print(f"Number of unique authors : {df_model['authors'].nunique()}")
print(f"Number of unique first authors : {df_model['first_author'].nunique()}")


#### Checking for rating patterns based on the number of authors

In [ ]:
# Compare the distribution of author counts and how ratings vary with the number of authors.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.countplot(
    data=df_model.assign(author_count_bucket=df_model["n_authors"].clip(upper=10).astype(int)),
    x="author_count_bucket",
    color="#2a9d8f",
    ax=axes[0],
)
axes[0].set_title("Authors per book (capped at 10 for readability)")
axes[0].set_xlabel("Number of authors")
axes[0].set_ylabel("Books")

high_author_count_share = (df_model["n_authors"] > 10).mean()
axes[0].text(
    0.98,
    0.95,
    f"{high_author_count_share:.2%} of rows have >10 authors",
    transform=axes[0].transAxes,
    ha="right",
    va="top",
    fontsize=9,
)

sns.boxplot(data=df_model, x="n_authors", y="average_rating",
            color="#e9c46a", ax=axes[1])
axes[1].set_title("Rating vs author count (raw counts)")
axes[1].set_xlabel("Number of authors")
plt.tight_layout()
plt.show()


No clear correlation between number of authors and average rating.


#### Modelling authors features


One simple way to keep some author information without one-hot encoding thousands of author names is to add `first_author_frequency`: how often the first author appears in `df_model`. Another way is to encode the average rating of the first author into the training set (target encoding)

#### Creating author frequency feature

In [ ]:
# Frequency encoding candidate for author information.
# Keep only one frequency feature: how many rows share the same first_author.
first_author_frequency_counts = df_model["first_author"].value_counts()

df_model["first_author_frequency"] = (
    df_model["first_author"].map(first_author_frequency_counts).fillna(0).astype(int)
)

author_feature_preview_columns = [
    "first_author",
    "authors",
    "n_authors",
    "first_author_frequency",
]

df_model[author_feature_preview_columns].head()


#### Author target encoding

Target encoding replaces a category with the average value of the target (average_ratings) for that category. Here we create an exploratory smoothed target mean for `first_author`.

#### Helper function for target encoding (Used for publisher and first_author)

In [ ]:
# Exploratory target encoding for first_author.
# Smoothing pulls rare authors toward the global mean so one-book authors are not over-trusted.
TARGET_ENCODING_SMOOTHING = 10

def add_smoothed_target_encoding(data, category_column, encoded_column, count_column):
    global_target_mean = data["average_rating"].mean()
    mean_column = f"{category_column}_mean_rating"

    target_summary = (
        data.groupby(category_column, dropna=False)
        .agg(
            **{
                count_column: ("average_rating", "size"),
                mean_column: ("average_rating", "mean"),
            }
        )
    )
    target_summary[encoded_column] = (
        (
            target_summary[count_column] * target_summary[mean_column]
            + TARGET_ENCODING_SMOOTHING * global_target_mean
        )
        / (target_summary[count_column] + TARGET_ENCODING_SMOOTHING)
    )

    data[encoded_column] = (
        data[category_column]
        .map(target_summary[encoded_column])
        .fillna(global_target_mean)
    )

    return target_summary.reset_index()

Applying the helper function to the first_author feature

In [ ]:
first_author_target_summary = add_smoothed_target_encoding(
    df_model,
    category_column="first_author",
    encoded_column="first_author_target_mean",
    count_column="author_books",
)

first_author_target_summary.sort_values(
    "author_books", ascending=False).head(15).round(3)

#### Check author features correlation with average_ratings

In [ ]:
# Spearman correlation with average_rating for the selected numeric author features.
author_numeric_features = [
    "n_authors",
    "first_author_frequency",
    "first_author_target_mean",
]

author_feature_correlation = (
    df_model[author_numeric_features + ["average_rating"]]
    .corr(method="spearman")
    .loc[author_numeric_features, "average_rating"]
    .reset_index()
    .rename(columns={"index": "feature", "average_rating": "spearman_corr"})
    .assign(abs_spearman=lambda data: data["spearman_corr"].abs())
    .sort_values("abs_spearman", ascending=False)
    .drop(columns="abs_spearman")
)

author_feature_correlation.round(4)

`n_authors` has only a small positive rank correlation with `average_rating`, and `first_author_frequency` is weaker. `first_author_target_mean` is more directly related to the target because it is built from average ratings, this is the most important feature for modelling so far, but it should be treated carefully: only the training set should be used to fit it.

### 2.9 `title`


#### Duplicates


We already established that there are no strict duplicates because `bookID`, `isbn`, and `isbn13` are unique. Repeated titles may instead represent different editions, languages, publishers, or collections.


#### Title duplicates

In [ ]:
# Reuse title frequencies throughout the duplicate-title analysis.
title_frequencies = df_model["title"].value_counts()
title_frequencies


#### Inspect one repeated title

Using `the iliad` as an example to see how editions and author strings differ for the same title.


In [ ]:
df_model[df_model["title"] == "the iliad"]

For this title, the author strings vary while the first author remains the same. This supports using `first_author` as a compact feature, but not as a unique identifier for an edition.

#### Duplicates by Title and authors

In [ ]:
# One consolidated summary for repeated titles and their author variants.
title_author_summary = (
    df_model.groupby("title")
    .agg(
        n_books=("title", "size"),
        n_author_strings=("authors", "nunique"),
        n_first_authors=("first_author", "nunique"),
    )
)

repeated_titles = title_frequencies[title_frequencies.gt(1)].index
titles_with_different_first_author = title_author_summary.index[
    title_author_summary["n_first_authors"].gt(1)
]
duplicate_title_author_pairs = df_model.groupby(["title", "authors"]).size().gt(1).sum()
duplicate_title_first_author_pairs = (
    df_model.groupby(["title", "first_author"]).size().gt(1).sum()
)

print(f"Repeated titles: {len(repeated_titles)}")
print(f"Repeated title + author pairs: {duplicate_title_author_pairs}")
print(f"Repeated title + first-author pairs: {duplicate_title_first_author_pairs}")
print("Repeated titles with more than one first author: "f"{len(titles_with_different_first_author)}")


#### Inspect title conflicts


In [ ]:
# Show the title/first-author combinations for repeated titles with multiple first authors.
title_first_author_groups = (
    df_model.loc[df_model["title"].isin(titles_with_different_first_author)]
    .groupby(["title", "first_author"])
    .size()
    .reset_index(name="count")
    .sort_values(["title", "count"], ascending=[True, False])
    .head(20)
)
title_first_author_groups

#### Inspect rating variants within same-title first-author groups

Aggregate each title/first-author group once to identify cases where the group contains different ratings.


In [ ]:
# Inspect same-title, same-first-author groups that have more than one rating.
title_first_author_rating_variants = (
    df_model.groupby(["title", "first_author"])
    .agg(
        n_unique_average_rating=("average_rating", "nunique"),
        ratings_list=("average_rating", lambda ratings: list(ratings.unique())),
        num_pages_list=("num_pages", lambda page_counts: list(page_counts.unique())),
        language_code_list=("language_code", lambda language_codes: list(language_codes.unique())),
        publisher_list=("publisher", lambda publishers: list(publishers.unique())),
        publication_year_list=("publication_year", lambda publication_years: list(publication_years.unique())),
    )
    .reset_index()
    .query("n_unique_average_rating > 1")
)

print(
    "Same-title, same-first-author groups with different average ratings: "
    f"{len(title_first_author_rating_variants)}"
)
title_first_author_rating_variants


The rating variants show that `title` plus `first_author` is not a reliable duplicate key: the records can differ in edition-level fields such as publisher, language, page count, and publication year.


**[Decision]** Keep these rows as they are. Repeated titles may represent different editions, publishers, languages, or collections.


#### Collections and series


We can see from the title strings that some books are part of a series or a collection. Lets define a pattern to identify these books.


In [ ]:
# Add title-based flags for collections and series entries.
# The identifiers for series are "#[number]" / "Volume [number]" / "Vol. [number]" / "Books [number]" / "Part [number]" / "Tome [number]".
TITLE_REGEX_FLAGS = re.IGNORECASE | re.VERBOSE

SERIES_NUMBER_PATTERN = r"(?:\d+(?:\.\d+)?|[ivxlcdm]+|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)"
SERIES_NUMBER_RANGE_PATTERN = rf"{SERIES_NUMBER_PATTERN}(?:\s*(?:[-\u2013\u2014,/&]|and)\s*{SERIES_NUMBER_PATTERN})*"

collection_title_pattern = r"""
\b(?:box(?:ed)?\s+set|omnibus|antholog(?:y|ies)|collection)\b
|
\b(?:complete|collected|selected|essential|major)\s+
(?:works|novels|stories|short\s+stories|poems|poetry|plays|writings|letters|essays|tragedies|adventures|memoirs)\b
|
\b\d+\s*(?:book|volume)s?\s+(?:box(?:ed)?\s+set|set|collection)\b
|
\b(?:\d+\s+)?volumes?\s+set\b
"""

series_title_pattern = rf"""
\([^)]*(?:\#\s*{SERIES_NUMBER_RANGE_PATTERN}|\b(?:vol(?:ume)?s?\.?|books?|part|tome)\s+{SERIES_NUMBER_RANGE_PATTERN})[^)]*\)
|
(?<!\w)\#\s*{SERIES_NUMBER_RANGE_PATTERN}\b
|
\b(?:vol(?:ume)?s?\.?|part|tome)\s+{SERIES_NUMBER_RANGE_PATTERN}\b
"""

title_series = df_model["title"].fillna("")

df_model["is_collection"] = title_series.str.contains(
    collection_title_pattern, regex=True, flags=TITLE_REGEX_FLAGS, na=False)
df_model["is_series"] = title_series.str.contains(
    series_title_pattern, regex=True, flags=TITLE_REGEX_FLAGS, na=False)


#### Series and collections average ratings

In [ ]:
# Compare the rating level of books flagged as collections or series entries.
# The comparison uses both mean and median because average_rating is bounded and slightly skewed.
collection_series_summary_rows = []

for label, flag_column in {
    "Collections": "is_collection",
    "Series entries": "is_series",
}.items():
    flagged_books = df_model[df_model[flag_column]]
    unflagged_books = df_model[~df_model[flag_column]]

    collection_series_summary_rows.append({
        "category": label,
        "books": len(flagged_books),
        "share_of_df_model": len(flagged_books) / len(df_model),
        "mean_rating": flagged_books["average_rating"].mean(),
        "median_rating": flagged_books["average_rating"].median(),
        "non_flagged_mean_rating": unflagged_books["average_rating"].mean(),
        "mean_difference": flagged_books["average_rating"].mean() - unflagged_books["average_rating"].mean(),
    })

collection_series_summary_df = pd.DataFrame(collection_series_summary_rows)
collection_series_summary_df.round(3)


Collections represent a small share of `df_model`, so their higher mean and median rating should be interpreted carefully. The pattern is still plausible: collections, complete works, boxed sets, and selected works are often created for books or authors with an existing audience.

Series entries are much more common. Their mean rating is only slightly higher than non-series books, so `is_series` may add some signal, but it is unlikely to be a strong standalone predictor.

### 2.10 `publisher`


#### First look

Publisher was already used earlier in the audiobook processing: `is_audio` was derived from title and publisher text because several low or zero page-count records looked like audiobook publishers.


#### Count publishers

Counting publisher frequencies to understand how sparse this categorical field is.


In [ ]:
publisher_value_counts = df_model["publisher"].value_counts(dropna=False)
publisher_value_counts.head(20)


#### Plot most common publishers

In [ ]:
TOP_PUBLISHER_PLOT_COUNT = 15

top_publisher_count_df = (
    publisher_value_counts
    .head(TOP_PUBLISHER_PLOT_COUNT)
    .rename_axis("publisher")
    .reset_index(name="books")
)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=top_publisher_count_df,
    y="publisher",
    x="books",
    color="#2a9d8f",
)
plt.title("Most frequent publishers")
plt.xlabel("Number of books")
plt.ylabel("Publisher")
plt.tight_layout()
plt.show()


The publisher column has many rare values, so one-hot encoding all publishers would create a sparse feature set.

#### Profile top publishers

In [ ]:
TOP_PUBLISHER_COUNT = 15

top_publisher_names = publisher_value_counts.head(TOP_PUBLISHER_COUNT).index

publisher_summary = (
    df_model[df_model["publisher"].isin(top_publisher_names)]
    .groupby("publisher")
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
        rating_std=("average_rating", "std"),
        audiobook_share=("is_audio", "mean"),
        collection_share=("is_collection", "mean"),
        series_share=("is_series", "mean"),
    )
    .sort_values("books", ascending=False)
    .reset_index()
)
publisher_summary["share_of_df_model"] = publisher_summary["books"] / len(df_model)

publisher_summary.round(3)


#### Publisher frequency feature

As with `first_author_frequency`, a simple publisher frequency encoding keeps broad publisher information without creating thousands of columns. This feature counts how often each publisher appears in `df_model`.


In [ ]:
publisher_frequency_counts = df_model["publisher"].value_counts()

df_model["publisher_frequency"] = (df_model["publisher"].map(publisher_frequency_counts).fillna(0).astype(int))

publisher_feature_preview_columns = [
    "publisher",
    "publisher_frequency",
    "average_rating",
]

df_model[publisher_feature_preview_columns].head()

#### Publisher target encoding

Similar to `first_author_target_mean`, the average rating for each publisher is used as a feature for the modelling pipeline (with smoothing for rare publishers).

In [ ]:
# Exploratory target encoding for publisher.
publisher_target_summary = add_smoothed_target_encoding(
    df_model,
    category_column="publisher",
    encoded_column="publisher_target_mean",
    count_column="publisher_books",
)

publisher_target_summary.sort_values("publisher_books", ascending=False).head(15).round(3)

#### Publisher feature correlation with average_rating

In [ ]:
# Spearman correlation with average_rating for the selected numeric publisher features.
publisher_numeric_features = [
    "publisher_frequency",
    "publisher_target_mean",
]

publisher_feature_correlation = (
    df_model[publisher_numeric_features + ["average_rating"]]
    .corr(method="spearman")
    .loc[publisher_numeric_features, "average_rating"]
    .reset_index()
    .rename(columns={"index": "feature", "average_rating": "spearman_corr"})
)

publisher_feature_correlation.round(4)

`publisher_frequency` is a simple candidate feature for the first modelling pass, but by itself it does not have a strong relationship with average rating. `publisher_target_mean` is more informative because it uses the publisher's historical average rating, although it is less of a signal than `first_author_target_mean`.

------------
## 3. Modelling

For this project, six regression models are evaluated:
- **Linear Regression** : A simple, interpretable model that assumes a linear relationship between the input features and the target variable (average rating).
- **HistGradientBoostingRegressor** : An efficient, tree-based ensemble algorithm from scikit-learn for regression tasks, well-suited to handling mixed feature types and capturing nonlinear relationships.
- **CatBoostRegressor** : A powerful, fast gradient boosting algorithm that naturally handles categorical features and often achieves strong performance with minimal preprocessing.
- **XGBRegressor** : A regularized gradient-boosted tree model with row and feature subsampling controls.
- **RandomForestRegressor** : A bagged ensemble that averages bootstrapped decision trees to reduce variance.
- **ExtraTreesRegressor** : A highly randomized tree ensemble that averages trees built with random split thresholds.

The models are evaluated with:
- **Mean Absolute Error (MAE)**: average absolute prediction error in rating points. Lower is better.
- **Root Mean Squared Error (RMSE)**: similar to MAE but penalizes larger errors more strongly. Lower is better.
- **R-squared (R2)**: share of target variance explained by the model. Higher is better.

This section keeps the full modelling pipeline visible in the notebook: modelling data preparation, feature engineering, feature selection, model training, tuning, and evaluation. A separate export process will later reproduce the selected final pipeline once the features and model to use are decided.


### 3.1 Modelling setup

#### Import modelling tools

In [ ]:
# Import the modelling tools used in this section.
# These imports stay in the notebook because the professor should be able to see the full
# modelling workflow without opening an external preprocessing file.
from itertools import product

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import plotly.graph_objects as go
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


### 3.2 Modelling source table

The EDA section created exploratory columns to understand the data. For final model training, we rebuild the input table from cleaned source columns so that train-fitted feature engineering can be applied after the train/test split.


The modelling table below applies the row filters justified in EDA. The feature engineering section then handles reusable input transformations.


#### Build the modelling source table

We define the target and rebuild the raw modelling inputs from cleaned source columns before any train/test split.


In [ ]:
TARGET = "average_rating"

# Build the raw modelling table from cleaned source columns only.
# The row filters match decisions justified in EDA: remove books with very few ratings
# `publication_year` is not included here because it is engineered from `publication_date` later.
RAW_MODEL_INPUT_COLUMNS = [
    "title",
    "authors",
    "language_code",
    "num_pages",
    "ratings_count",
    "text_reviews_count",
    "publication_date",
    "publisher",
]
MIN_RATINGS_COUNT_FOR_MODELLING = 50

model_source_df = df_clean.copy()
model_source_df = model_source_df[model_source_df["ratings_count"] >= MIN_RATINGS_COUNT_FOR_MODELLING].copy()

X_model_raw = model_source_df[RAW_MODEL_INPUT_COLUMNS].copy()
y_model = model_source_df[TARGET].copy()

model_dataset_summary = pd.DataFrame([
    {"metric": "rows", "value": len(model_source_df)},
    {"metric": "raw_input_columns", "value": len(RAW_MODEL_INPUT_COLUMNS)},
    {"metric": "minimum_ratings_count", "value": MIN_RATINGS_COUNT_FOR_MODELLING},
])

model_dataset_summary


### 3.3 Train/test split


#### Create the train/test split

We set the split parameters and use rating bands so train and test keep a similar target distribution.


In [ ]:
# Create one fixed stratified split for every model configuration.
# Stratification keeps the concentrated average_rating distribution similar in train and test.
TEST_SIZE = 0.15
RATING_BAND_COUNT = 10
RANDOM_STATE = 42

def create_stratified_rating_split(target):
    rating_band_intervals = pd.qcut(
        target,
        q=RATING_BAND_COUNT,
        duplicates="drop",
    )
    rating_band_codes = rating_band_intervals.cat.codes

    train_index, test_index = train_test_split(
        target.index,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=rating_band_codes,
    )

    return train_index, test_index, rating_band_intervals


MODEL_TRAIN_INDEX, MODEL_TEST_INDEX, model_rating_bands = create_stratified_rating_split(y_model)

split_check_df = pd.concat([
    pd.DataFrame({
        "split": "train",
        TARGET: y_model.loc[MODEL_TRAIN_INDEX],
        "rating_band": model_rating_bands.loc[MODEL_TRAIN_INDEX].astype(str),
    }),
    pd.DataFrame({
        "split": "test",
        TARGET: y_model.loc[MODEL_TEST_INDEX],
        "rating_band": model_rating_bands.loc[MODEL_TEST_INDEX].astype(str),
    }),
])

split_summary = (
    split_check_df
    .groupby("split")
    .agg(
        books=(TARGET, "size"),
        mean_rating=(TARGET, "mean"),
        median_rating=(TARGET, "median"),
        min_rating=(TARGET, "min"),
        max_rating=(TARGET, "max"),
        rating_std=(TARGET, "std"),
    )
    .reindex(["train", "test"])
)
split_summary["share_of_data"] = split_summary["books"] / len(model_source_df)

split_summary.round(3)

#### Check split balance

We compare the percentage of each rating band in train and test to verify that stratification worked.


In [ ]:
# Check whether each rating band has a similar share in train and test.
rating_band_split_share = pd.crosstab(
    split_check_df["rating_band"],
    split_check_df["split"],
    normalize="columns",
).reindex(columns=["train", "test"])

(rating_band_split_share * 100).round(1)


### 3.4 Feature Description

The final modelling feature set keeps features that are simple to explain and available from the Goodreads-style book metadata. Some feature ideas use different column versions depending on the model: linear regression receives log-scaled versions of skewed count features, while tree-based models receive the raw numeric values.

| Feature / feature idea | Rationale | Model use |
|---|---|---|
| `ratings_count` / `log_ratings_count` | Rating volume shows how much support the average rating has. More ratings generally make the average rating more stable. | Log version for Linear Regression; raw version for tree models. |
| `text_reviews_count` / `log_text_reviews_count` | Written reviews are a stronger engagement signal than simple star ratings. | Log version for Linear Regression; raw version for tree models. |
| `review_to_rating_ratio` / `log_review_share` | Measures review intensity: among users who rated the book, how many wrote a text review. | Log version for Linear Regression; raw version for tree models. |
| `num_pages` / `log_num_pages` | Page count captures book length, which may relate to genre, format, or reader commitment. | Log version for Linear Regression; raw version for tree models. |
| `num_pages_missing` | Flags books where page count was missing or invalid before imputation, especially useful for audiobook-like records. | All models. |
| `book_age` | Older books have had more time to accumulate readers and may reflect classics or long-lived popularity. | All models. |
| `publication_decade` | Groups publication timing into broader historical periods without treating every exact year as a separate category. | One-hot encoded for sklearn models; categorical for CatBoost. |
| `language_code_grouped` | Keeps frequent language codes while grouping rare languages into `other_language`, avoiding a sparse feature. | One-hot encoded for sklearn models; categorical for CatBoost. |
| `n_authors` | Multiple authors can indicate anthologies, edited works, collaborations, or collections. | All models. |
| `first_author` | The main author can carry strong book-quality and audience information, but it is high-cardinality. | CatBoost only, because it can handle categorical features directly. |
| `first_author_frequency` / `log_first_author_frequency` | Counts how often the first author appears in the training data, giving a compact familiarity signal. | Log version for Linear Regression; raw version for tree models. |
| `first_author_target_mean` | Smoothed average rating for the first author, fitted on the training set only. | Sklearn tree models. |
| `first_author_rating_std` | Smoothed rating variability across the first author's books. | Sklearn tree models. |
| `publisher` | Publisher identity can reflect editorial segment, format, or catalogue type, but it is high-cardinality. | CatBoost only, because it can handle categorical features directly. |
| `publisher_frequency` / `log_publisher_frequency` | Counts how often the publisher appears in the training data, keeping a compact publisher signal. | Log version for Linear Regression; raw version for tree models. |
| `publisher_target_mean` | Smoothed average rating for the publisher, fitted on the training set only. | Sklearn tree models. |
| `publisher_rating_std` | Smoothed rating variability across the publisher's books. | Sklearn tree models. |
| `title_character_count` / `title_word_count` | Compact measures of title length and structure. | All models. |
| `is_audio` | Audiobook or audio-related records often behave differently for page count and reader engagement. | All models. |
| `is_collection` | Collections, boxed sets, complete works, and anthologies may have different rating behavior from single books. | All models. |
| `is_series` | Series entries may benefit from existing audience selection and continuity effects. | All models. |

### 3.5 Feature Engineering

This is the final feature engineering used by the models. The design follows the EDA findings above, but it is implemented here as a train-fitted pipeline:
- text fields are normalized again so future web-app inputs can be handled consistently;
- `publication_year`, `book_age`, and `publication_decade` are derived from the raw `publication_date` field;
- `num_pages = 0` is treated as missing, with a missing flag and train-fitted median imputation;
- author and publisher frequencies are learned from the training set only;
- author and publisher target means and rating variability are learned from the training set only, with unseen values falling back to global training statistics;
- common language-code groups are learned from the training set only;
- review engagement is captured as text reviews divided by total ratings;
- collection and series indicators are derived from the title text;
- one compact set of count, page, date, author, publisher, language, engagement, and title-structure features is then created consistently for train, test, and manual prediction inputs.

#### Set feature-engineering patterns

In [ ]:
# Constants used by the notebook feature engineering pipeline.
# They are repeated here instead of relying on EDA variables so this modelling section is readable on its own.
REFERENCE_YEAR = 2026
MIN_LANGUAGE_COUNT = 50
TARGET_ENCODING_SMOOTHING = 10

AUDIO_INDICATOR_PATTERN = r"""
audio(?:books?|go|text)?\b
|
\bcassettes?\b
|
\bcd\s*audio\b
|
\baudio\s*cd\b
|
\bmp3\b
|
\bsound\s+recording\b
|
\blistening\s+library\b
"""

SERIES_NUMBER_PATTERN = r"(?:\d+(?:\.\d+)?|[ivxlcdm]+|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)"
SERIES_NUMBER_RANGE_PATTERN = rf"{SERIES_NUMBER_PATTERN}(?:\s*(?:[-\u2013\u2014,/&]|and)\s*{SERIES_NUMBER_PATTERN})*"

COLLECTION_TITLE_PATTERN = r"""
\b(?:box(?:ed)?\s+set|omnibus|antholog(?:y|ies)|collection)\b
|
\b(?:complete|collected|selected|essential|major)\s+
(?:works|novels|stories|short\s+stories|poems|poetry|plays|writings|letters|essays|tragedies|adventures|memoirs)\b
|
\b\d+\s*(?:book|volume)s?\s+(?:box(?:ed)?\s+set|set|collection)\b
|
\b(?:\d+\s+)?volumes?\s+set\b
"""

SERIES_TITLE_PATTERN = rf"""
\([^)]*(?:\#\s*{SERIES_NUMBER_RANGE_PATTERN}|\b(?:vol(?:ume)?s?\.?|books?|part|tome)\s+{SERIES_NUMBER_RANGE_PATTERN})[^)]*\)
|
(?<!\w)\#\s*{SERIES_NUMBER_RANGE_PATTERN}\b
|
\b(?:vol(?:ume)?s?\.?|part|tome)\s+{SERIES_NUMBER_RANGE_PATTERN}\b
"""

#### Input and normalization helpers

In [ ]:
# Helper functions for feature engineering.
def normalize_model_text(value: Any) -> str | None:
    """Normalize a single text value for stable feature keys."""
    if value is None or pd.isna(value):
        return None

    normalized_text = str(value).strip()
    if not normalized_text:
        return ""

    # The frequency mappings should not split the same author/publisher because of case or accents.
    normalized_text = normalized_text.lower()
    normalized_text = unicodedata.normalize("NFKD", normalized_text)
    normalized_text = "".join(
        character
        for character in normalized_text
        if not unicodedata.combining(character)
    )
    normalized_text = re.sub(r"\s+", " ", normalized_text)
    return normalized_text


def series_or_default(frame, column, default_value):
    """Return one column or a default-valued Series for manual/web-app style inputs."""
    if column in frame.columns:
        return frame[column]
    return pd.Series(default_value, index=frame.index)


def numeric_series(frame, column, default_value=0):
    """Coerce numeric inputs while keeping invalid values as missing."""
    return pd.to_numeric(series_or_default(frame, column, default_value), errors="coerce")


def normalized_text_series(frame, column, missing_value):
    """Normalize text columns and keep an explicit missing bucket."""
    values = series_or_default(frame, column, None).map(normalize_model_text)
    return values.fillna(missing_value).replace("", missing_value)


#### Smoothed target-encoding helper

These helpers learn smoothed category means and rating variability from the training target.

In [ ]:
def fit_smoothed_target_encoding(category_values, target_values):
    """Learn smoothed target means for one categorical feature."""
    encoding_data = pd.DataFrame({
        "category": category_values,
        "target": pd.to_numeric(target_values, errors="coerce"),
    }).dropna(subset=["target"])

    global_target_mean = encoding_data["target"].mean()
    category_summary = encoding_data.groupby("category", dropna=False)["target"].agg(["count", "mean"])

    smoothed_target_mean = (
        category_summary["count"] * category_summary["mean"]
        + TARGET_ENCODING_SMOOTHING * global_target_mean
    ) / (category_summary["count"] + TARGET_ENCODING_SMOOTHING)

    return smoothed_target_mean


def fit_smoothed_target_std_encoding(category_values, target_values):
    """Learn category rating variability, shrunk toward the global variance."""
    encoding_data = pd.DataFrame({
        "category": category_values,
        "target": pd.to_numeric(target_values, errors="coerce"),
    }).dropna(subset=["target"])

    global_variance = encoding_data["target"].var(ddof=0)
    grouped_target = encoding_data.groupby("category", dropna=False)["target"]
    category_count = grouped_target.count()
    category_variance = grouped_target.var(ddof=0).fillna(0)

    smoothed_variance = (
        category_count * category_variance
        + TARGET_ENCODING_SMOOTHING * global_variance
    ) / (category_count + TARGET_ENCODING_SMOOTHING)

    return np.sqrt(smoothed_variance.clip(lower=0))


#### Prepare consistent base features

Raw notebook rows, alternative datasets, and future web-form rows all pass through the same visible cleaning steps. Missing columns receive explicit defaults and invalid numeric or date values are coerced safely.

In [ ]:
def prepare_base_model_features(raw_data):
    """Prepare raw input columns before deriving model features."""
    frame = raw_data.copy()
    base_features = pd.DataFrame(index=frame.index)

    # Normalize text first, because later author, publisher, and title flags depend on stable strings.
    base_features["title"] = normalized_text_series(frame, "title", "missing_title")
    base_features["authors"] = normalized_text_series(frame, "authors", "missing_author")
    base_features["publisher"] = normalized_text_series(frame, "publisher", "missing_publisher")
    base_features["language_code"] = (
        series_or_default(frame, "language_code", "missing_language")
        .fillna("missing_language")
        .replace("", "missing_language")
        .astype(str)
    )

    # Counts are clipped at zero so a future bad input cannot create negative engagement features.
    base_features["ratings_count"] = numeric_series(frame, "ratings_count", 0).fillna(0).clip(lower=0)
    base_features["text_reviews_count"] = numeric_series(frame, "text_reviews_count", 0).fillna(0).clip(lower=0)
    base_features["num_pages_raw"] = numeric_series(frame, "num_pages", np.nan)

    publication_dates = pd.to_datetime(series_or_default(frame, "publication_date", pd.NaT), errors="coerce")
    base_features["publication_year"] = publication_dates.dt.year

    title_publisher_text = base_features["title"].fillna("") + " " + base_features["publisher"].fillna("")
    base_features["is_audio"] = title_publisher_text.str.contains(
        AUDIO_INDICATOR_PATTERN,
        regex=True,
        flags=re.IGNORECASE | re.VERBOSE,
        na=False,
    ).astype(int)

    # We keep the first author as a compact author signal and count all listed authors separately.
    base_features["first_author"] = (
        base_features["authors"]
        .str.split("/", n=1)
        .str[0]
        .str.strip()
        .replace("", "missing_author")
        .fillna("missing_author")
    )
    base_features["n_authors"] = (
        base_features["authors"]
        .str.split("/")
        .str.len()
        .fillna(1)
        .astype(float)
    )

    return base_features

#### Fit train-only feature references

In [ ]:
# Fit the feature-engineering reference values on the training rows only.
def fit_feature_engineering_reference(raw_train_data, train_target):
    base_train = prepare_base_model_features(raw_train_data)
    train_target = pd.to_numeric(train_target.reindex(base_train.index), errors="coerce")

    # Medians are learned from train only, so the test set does not influence imputation.
    publication_year_median = base_train["publication_year"].median()
    if pd.isna(publication_year_median):
        publication_year_median = REFERENCE_YEAR

    valid_page_counts = base_train["num_pages_raw"].where(base_train["num_pages_raw"] > 0)
    global_page_median = valid_page_counts.median()
    if pd.isna(global_page_median):
        global_page_median = 300

    page_medians_by_audio = {}
    for audio_value in [0, 1]:
        group_median = valid_page_counts[base_train["is_audio"] == audio_value].median()
        if pd.isna(group_median):
            group_median = global_page_median
        page_medians_by_audio[audio_value] = float(group_median)

    # These mappings are fitted on train only to avoid leakage from the test set.
    language_counts = base_train["language_code"].value_counts(dropna=False)
    common_languages = set(language_counts[language_counts >= MIN_LANGUAGE_COUNT].index)
    global_target_mean = train_target.mean()
    global_target_std = train_target.std(ddof=0)

    return {
        "publication_year_median": float(publication_year_median),
        "global_page_median": float(global_page_median),
        "page_medians_by_audio": page_medians_by_audio,
        "first_author_frequency_counts": base_train["first_author"].value_counts(dropna=False),
        "publisher_frequency_counts": base_train["publisher"].value_counts(dropna=False),
        "target_global_mean": float(global_target_mean),
        "target_global_std": float(global_target_std),
        "first_author_target_mean": fit_smoothed_target_encoding(base_train["first_author"], train_target),
        "publisher_target_mean": fit_smoothed_target_encoding(base_train["publisher"], train_target),
        "first_author_rating_std": fit_smoothed_target_std_encoding(base_train["first_author"], train_target),
        "publisher_rating_std": fit_smoothed_target_std_encoding(base_train["publisher"], train_target),
        "common_languages": common_languages,
    }

#### Build core model features

These features use the normalized raw values and train-fitted date and page-count references.


In [ ]:
def build_core_model_features(base_features, feature_reference):
    """Create date, engagement, page-count, and author-count features."""
    engineered = pd.DataFrame(index=base_features.index)

    engineered["first_author"] = base_features["first_author"]
    engineered["publisher"] = base_features["publisher"]

    # Date features use the train median when publication_date is missing or invalid.
    publication_year = base_features["publication_year"].fillna(feature_reference["publication_year_median"])
    engineered["book_age"] = (REFERENCE_YEAR - publication_year).clip(lower=0)
    engineered["publication_decade"] = ((publication_year.astype(int) // 10) * 10).astype(str) + "s"

    # Engagement features keep rating volume and the share of ratings that produced text reviews.
    engineered["ratings_count"] = base_features["ratings_count"]
    engineered["text_reviews_count"] = base_features["text_reviews_count"]
    ratings_for_ratio = engineered["ratings_count"].replace(0, np.nan)
    engineered["review_to_rating_ratio"] = (engineered["text_reviews_count"] / ratings_for_ratio).fillna(0)
    engineered["log_review_share"] = np.log1p(engineered["review_to_rating_ratio"].clip(lower=0))
    engineered["is_audio"] = base_features["is_audio"].astype(int)

    # Page counts of zero are treated as missing, then imputed by audiobook/non-audiobook median.
    engineered["num_pages_missing"] = (base_features["num_pages_raw"].isna() | base_features["num_pages_raw"].le(0)).astype(int)
    engineered["num_pages"] = base_features["num_pages_raw"].where(base_features["num_pages_raw"] > 0).astype(float)
    for audio_value, median_page_count in feature_reference["page_medians_by_audio"].items():
        group_mask = engineered["is_audio"].eq(audio_value)
        engineered.loc[group_mask, "num_pages"] = engineered.loc[group_mask, "num_pages"].fillna(median_page_count)
    engineered["num_pages"] = engineered["num_pages"].fillna(feature_reference["global_page_median"])

    engineered["n_authors"] = base_features["n_authors"].fillna(1).clip(lower=1)
    return engineered

#### Add frequency and target-statistic encodings

The mappings come only from the fitted reference. Unseen categories receive zero frequency and global training-target statistics.

In [ ]:
def add_reference_encoded_features(engineered, base_features, feature_reference):
    """Add train-fitted frequency, target statistics, and language features."""
    engineered["first_author_frequency"] = (
        base_features["first_author"]
        .map(feature_reference["first_author_frequency_counts"])
        .fillna(0)
        .astype(int)
    )
    engineered["publisher_frequency"] = (
        base_features["publisher"]
        .map(feature_reference["publisher_frequency_counts"])
        .fillna(0)
        .astype(int)
    )

    engineered["first_author_target_mean"] = (
        base_features["first_author"]
        .map(feature_reference["first_author_target_mean"])
        .fillna(feature_reference["target_global_mean"])
        .astype(float)
    )
    engineered["publisher_target_mean"] = (
        base_features["publisher"]
        .map(feature_reference["publisher_target_mean"])
        .fillna(feature_reference["target_global_mean"])
        .astype(float)
    )
    engineered["first_author_rating_std"] = (
        base_features["first_author"]
        .map(feature_reference["first_author_rating_std"])
        .fillna(feature_reference["target_global_std"])
        .astype(float)
    )
    engineered["publisher_rating_std"] = (
        base_features["publisher"]
        .map(feature_reference["publisher_rating_std"])
        .fillna(feature_reference["target_global_std"])
        .astype(float)
    )

    engineered["language_code_grouped"] = base_features["language_code"].where(
        base_features["language_code"].isin(feature_reference["common_languages"]),
        "other_language",
    )
    return engineered


#### Add title features and log transformations

In [ ]:
def add_title_and_log_features(engineered, base_features):
    """Add title-derived counts and flags plus log-transformed features."""
    title_text = base_features["title"].where(base_features["title"].ne("missing_title"), "")
    engineered["title_character_count"] = title_text.str.len().fillna(0).astype(int)
    engineered["title_word_count"] = title_text.str.split().str.len().fillna(0).astype(int)
    engineered["is_collection"] = title_text.str.contains(
        COLLECTION_TITLE_PATTERN,
        regex=True,
        flags=re.IGNORECASE | re.VERBOSE,
        na=False,
    ).astype(int)
    engineered["is_series"] = title_text.str.contains(
        SERIES_TITLE_PATTERN,
        regex=True,
        flags=re.IGNORECASE | re.VERBOSE,
        na=False,
    ).astype(int)

    engineered["log_ratings_count"] = np.log1p(engineered["ratings_count"])
    engineered["log_text_reviews_count"] = np.log1p(engineered["text_reviews_count"])
    engineered["log_num_pages"] = np.log1p(engineered["num_pages"])
    engineered["log_first_author_frequency"] = np.log1p(engineered["first_author_frequency"])
    engineered["log_publisher_frequency"] = np.log1p(engineered["publisher_frequency"])
    return engineered


#### Apply the complete feature-engineering sequence

This small orchestrator makes the order of the three transformation stages explicit for training, testing, alternative datasets, and future web inputs.

In [ ]:
def apply_feature_engineering(raw_data, feature_reference):
    """Apply the complete fitted feature-engineering sequence."""
    base_features = prepare_base_model_features(raw_data)
    engineered = build_core_model_features(base_features, feature_reference)
    engineered = add_reference_encoded_features(engineered, base_features, feature_reference)
    engineered = add_title_and_log_features(engineered, base_features)
    return engineered


#### Create engineered train and test tables

We fit feature engineering on the training raw rows and apply the result separately to train and test.


In [ ]:
# Fit feature engineering on train only, then apply it to train and test.
feature_engineering_reference = fit_feature_engineering_reference(
    X_model_raw.loc[MODEL_TRAIN_INDEX],
    y_model.loc[MODEL_TRAIN_INDEX],
)

X_train_engineered = apply_feature_engineering(
    X_model_raw.loc[MODEL_TRAIN_INDEX],
    feature_engineering_reference,
)
X_test_engineered = apply_feature_engineering(
    X_model_raw.loc[MODEL_TEST_INDEX],
    feature_engineering_reference,
)

feature_engineering_summary = pd.DataFrame([
    {"metric": "train_rows", "value": len(X_train_engineered)},
    {"metric": "test_rows", "value": len(X_test_engineered)},
    {"metric": "engineered_columns", "value": X_train_engineered.shape[1]},
    {"metric": "common_languages_learned", "value": len(feature_engineering_reference["common_languages"])},
    {"metric": "target_encoding_smoothing", "value": TARGET_ENCODING_SMOOTHING},
    {"metric": "author_target_mappings", "value": len(feature_engineering_reference["first_author_target_mean"])},
    {"metric": "publisher_target_mappings", "value": len(feature_engineering_reference["publisher_target_mean"])},
    {"metric": "author_std_mappings", "value": len(feature_engineering_reference["first_author_rating_std"])},
    {"metric": "publisher_std_mappings", "value": len(feature_engineering_reference["publisher_rating_std"])},
])

feature_engineering_summary

#### Preview engineered features

In [ ]:
# Inspect a few engineered rows so the transformation is visible in the notebook.
engineered_preview_columns = [
    "book_age",
    "num_pages",
    "num_pages_missing",
    "ratings_count",
    "text_reviews_count",
    "review_to_rating_ratio",
    "first_author",
    "first_author_frequency",
    "first_author_target_mean",
    "first_author_rating_std",
    "publisher_frequency",
    "publisher_target_mean",
    "publisher_rating_std",
    "language_code_grouped",
    "publication_decade",
    "title_character_count",
    "title_word_count",
    "is_audio",
    "is_collection",
    "is_series",
]

X_train_engineered[engineered_preview_columns].head()

### 3.6 Feature Selection

The feature lists combine shared features with small family-specific groups: linear regression receives log-transformed skewed values, sklearn tree models receive raw values and compact target statistics, and CatBoost receives raw categorical metadata.


#### Select features by model family

In [ ]:
# Shared structural features are used by every model family.
SHARED_NUMERIC_FEATURES = [
    "book_age",
    "n_authors",
    "num_pages_missing",
    "title_character_count",
    "title_word_count",
    "is_audio",
    "is_collection",
    "is_series",
]

LOW_CARDINALITY_CATEGORICAL_FEATURES = [
    "language_code_grouped",
    "publication_decade",
]

# Linear regression uses log transformations for skewed counts.
LINEAR_NUMERIC_FEATURES = [
    "log_ratings_count",
    "log_text_reviews_count",
    "log_review_share",
    "log_num_pages",
    "log_first_author_frequency",
    "log_publisher_frequency",
    *SHARED_NUMERIC_FEATURES,
]

# Sklearn tree models share raw values and compact author/publisher target statistics.
TREE_NUMERIC_FEATURES = [
    "ratings_count",
    "text_reviews_count",
    "review_to_rating_ratio",
    "num_pages",
    "first_author_frequency",
    "publisher_frequency",
    "first_author_target_mean",
    "publisher_target_mean",
    "first_author_rating_std",
    "publisher_rating_std",
    *SHARED_NUMERIC_FEATURES,
]

# CatBoost uses raw categorical identities, so it only needs non-target numeric aggregates.
CATBOOST_NUMERIC_FEATURES = [
    "ratings_count",
    "text_reviews_count",
    "review_to_rating_ratio",
    "num_pages",
    "first_author_frequency",
    "publisher_frequency",
    *SHARED_NUMERIC_FEATURES,
]
CATBOOST_CATEGORICAL_FEATURES = [
    "first_author",
    "publisher",
    *LOW_CARDINALITY_CATEGORICAL_FEATURES,
]

def make_model_spec(
    numeric_features,
    categorical_features,
    reported_parameters,
    *,
    fit_strategy="sklearn_pipeline",
    scale_numeric=False,
):
    """Build one compact model-family feature specification."""
    return {
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "fit_strategy": fit_strategy,
        "scale_numeric": scale_numeric,
        "reported_parameters": reported_parameters,
    }


# Each spec contains everything required by the shared training helper.
MODEL_SPECS = {
    "Linear regression": make_model_spec(
        LINEAR_NUMERIC_FEATURES, LOW_CARDINALITY_CATEGORICAL_FEATURES, [], scale_numeric=True
    ),
    "HistGradientBoosting": make_model_spec(
        TREE_NUMERIC_FEATURES,
        LOW_CARDINALITY_CATEGORICAL_FEATURES,
        ["max_iter", "learning_rate", "max_leaf_nodes", "l2_regularization", "min_samples_leaf"],
    ),
    "XGBoost": make_model_spec(
        TREE_NUMERIC_FEATURES,
        LOW_CARDINALITY_CATEGORICAL_FEATURES,
        ["n_estimators", "learning_rate", "max_depth", "min_child_weight"],
    ),
    "Random Forest": make_model_spec(
        TREE_NUMERIC_FEATURES,
        LOW_CARDINALITY_CATEGORICAL_FEATURES,
        ["n_estimators", "max_features", "min_samples_leaf"],
    ),
    "Extra Trees": make_model_spec(
        TREE_NUMERIC_FEATURES,
        LOW_CARDINALITY_CATEGORICAL_FEATURES,
        ["n_estimators", "max_features", "min_samples_leaf"],
    ),
    "CatBoost": make_model_spec(
        CATBOOST_NUMERIC_FEATURES,
        CATBOOST_CATEGORICAL_FEATURES,
        ["iterations", "learning_rate", "depth", "l2_leaf_reg"],
        fit_strategy="native_categorical",
    ),
}

model_feature_overview = pd.DataFrame([
    {
        "model": model_name,
        "fit_strategy": model_spec["fit_strategy"],
        "numeric_features": len(model_spec["numeric_features"]),
        "categorical_features": len(model_spec["categorical_features"]),
        "total_input_columns": (
            len(model_spec["numeric_features"])
            + len(model_spec["categorical_features"])
        ),
    }
    for model_name, model_spec in MODEL_SPECS.items()
])

model_feature_overview

#### Validate selected feature names

We check that every configured feature is present in the engineered training table before fitting models.


In [ ]:
# Confirm that every selected feature exists in the engineered training table.
selected_feature_names = sorted({
    feature_name
    for model_spec in MODEL_SPECS.values()
    for feature_name in (
        model_spec["numeric_features"]
        + model_spec["categorical_features"]
    )
})

missing_selected_features = [
    feature_name
    for feature_name in selected_feature_names
    if feature_name not in X_train_engineered.columns
]

pd.DataFrame({
    "selected_features": [len(selected_feature_names)],
    "missing_selected_features": [missing_selected_features],
})


### 3.7 Active Model Training

#### Defining active model configurations

In [ ]:
# Active model configurations used for normal notebook runs.
active_model_configs = [
    {
        "model": "Linear regression",
        "estimator": LinearRegression(),
    },
    {
        "model": "HistGradientBoosting", # tuning done
        "estimator": HistGradientBoostingRegressor(
            max_iter=75,
            learning_rate=0.13,
            max_leaf_nodes=63,
            l2_regularization=1,
            min_samples_leaf=20,
            early_stopping=False,
            random_state=RANDOM_STATE,
        ),
    },
    {
        "model": "XGBoost", # tuning done
        "estimator": XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            min_child_weight=6,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=5.0,
            objective="reg:squarederror",
            eval_metric="mae",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    },
    {
        "model": "Random Forest", # tuning done
        "estimator": RandomForestRegressor(
            n_estimators=200,
            max_features=0.8,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    },
    {
        "model": "Extra Trees", # tuning done
        "estimator": ExtraTreesRegressor(
            n_estimators=200,
            max_features=0.7,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    },
    {
        "model": "CatBoost",
        "estimator": CatBoostRegressor(
            iterations=400,
            learning_rate=0.09,
            depth=5,
            l2_leaf_reg=3,
            loss_function="RMSE",
            eval_metric="MAE",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        ),
    },
]

#### Training helpers

In [ ]:
# Training helpers used by the active and tuning model runs.
def make_one_hot_encoder():
    """Create a OneHotEncoder that works across recent sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def prepare_feature_matrix(engineered_data, numeric_features, categorical_features):
    """Select the engineered columns required by one model family."""
    feature_names = numeric_features + categorical_features
    X = engineered_data[feature_names].copy()

    bool_columns = X.select_dtypes(include="bool").columns
    X[bool_columns] = X[bool_columns].astype(int)

    for feature_name in numeric_features:
        X[feature_name] = pd.to_numeric(X[feature_name], errors="coerce")

    for feature_name in categorical_features:
        X[feature_name] = X[feature_name].astype("object").where(X[feature_name].notna(), "Missing").astype(str)

    return X


def build_sklearn_pipeline(model, numeric_features, categorical_features, scale_numeric=False):
    """Build preprocessing plus any sklearn-compatible regressor."""
    if not numeric_features and not categorical_features:
        raise ValueError("At least one numeric or categorical feature is required.")

    transformers = []

    if numeric_features:
        numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
        if scale_numeric:
            numeric_steps.append(("scaler", StandardScaler()))
        transformers.append(("num", Pipeline(steps=numeric_steps), numeric_features))

    if categorical_features:
        categorical_transformer = Pipeline(steps=[
            ("onehot", make_one_hot_encoder()),
        ])
        transformers.append(("cat", categorical_transformer, categorical_features))

    return Pipeline(steps=[
        ("preprocessor", ColumnTransformer(transformers=transformers)),
        ("model", clone(model)),
    ])


def model_parameters_for_result(model_name, estimator):
    """Extract the model settings shown in result tables and model lookups."""
    estimator_params = estimator.get_params()
    return {
        parameter_name: estimator_params[parameter_name]
        for parameter_name in MODEL_SPECS[model_name]["reported_parameters"]
        if parameter_name in estimator_params
    }


def format_parameters(parameters):
    """Create a compact readable label for notebook tables and model lookup keys, one per line."""
    if not parameters:
        return "default"
    return ", ".join(f"{key}={value}" for key, value in parameters.items())


def regression_metrics(y_true, y_pred):
    """Compute the regression metrics used in the project report."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


#### Evaluate model configurations

We fit each active configuration on the fixed split and collect metrics and fitted models.


In [ ]:
def fit_model_configuration(config, engineered_data, target_values):
    """Fit one configured model on any engineered dataset."""
    model_name = config["model"]
    model_spec = MODEL_SPECS[model_name]
    numeric_features = model_spec["numeric_features"]
    categorical_features = model_spec["categorical_features"]
    X = prepare_feature_matrix(engineered_data, numeric_features, categorical_features)

    if model_spec["fit_strategy"] == "native_categorical":
        fitted_model = clone(config["estimator"])
        fitted_model.fit(X, target_values, cat_features=categorical_features)
    elif model_spec["fit_strategy"] == "sklearn_pipeline":
        fitted_model = build_sklearn_pipeline(
            config["estimator"],
            numeric_features,
            categorical_features,
            scale_numeric=model_spec["scale_numeric"],
        )
        fitted_model.fit(X, target_values)
    else:
        raise ValueError(f"Unknown fit strategy for {model_name!r}: {model_spec['fit_strategy']}")

    model_parameters = model_parameters_for_result(model_name, config["estimator"])
    return {
        "model": fitted_model,
        "model_name": model_name,
        "parameters": format_parameters(model_parameters),
        "numeric_features": list(numeric_features),
        "categorical_features": list(categorical_features),
    }


def fit_and_evaluate_model(config):
    """Fit one model configuration and return metrics plus the fitted model."""
    model_name = config["model"]
    y_train = y_model.loc[MODEL_TRAIN_INDEX]
    y_test = y_model.loc[MODEL_TEST_INDEX]

    fitted_record = fit_model_configuration(config, X_train_engineered, y_train)
    X_test = prepare_feature_matrix(
        X_test_engineered,
        fitted_record["numeric_features"],
        fitted_record["categorical_features"],
    )
    y_pred = np.clip(fitted_record["model"].predict(X_test), 0, 5)
    metrics = regression_metrics(y_test, y_pred)

    result_row = {
        "model": model_name,
        "n_numeric_features": len(fitted_record["numeric_features"]),
        "n_categorical_features": len(fitted_record["categorical_features"]),
        "parameters": fitted_record["parameters"],
    }
    result_row.update(metrics)

    fitted_key = (model_name, fitted_record["parameters"])
    return result_row, fitted_key, fitted_record


def evaluate_model_configs(configs):
    """Evaluate several configurations and keep fitted models for prediction checks."""
    rows = []
    fitted_models = {}

    for config in configs:
        row, fitted_key, fitted_record = fit_and_evaluate_model(config)
        rows.append(row)
        fitted_models[fitted_key] = fitted_record

    return pd.DataFrame(rows), fitted_models


def fit_and_evaluate_constant_baseline():
    """Evaluate a mean DummyRegressor without using the feature pipeline."""
    y_train = y_model.loc[MODEL_TRAIN_INDEX]
    y_test = y_model.loc[MODEL_TEST_INDEX]
    X_train_constant = np.ones((len(y_train), 1))
    X_test_constant = np.ones((len(y_test), 1))

    baseline_model = DummyRegressor(strategy="mean")
    baseline_model.fit(X_train_constant, y_train)
    baseline_predictions = np.clip(baseline_model.predict(X_test_constant), 0, 5)
    baseline_metrics = regression_metrics(y_test, baseline_predictions)

    baseline_result = {
        "model": "Constant mean baseline",
        "n_numeric_features": 0,
        "n_categorical_features": 0,
        "parameters": "strategy=mean",
    }
    baseline_result.update(baseline_metrics)
    return baseline_result


#### Define prediction helper

In [ ]:
# The fitted reference is explicit so the same helper works for evaluation and exported models.
def predict_from_raw_input(fitted_record, raw_data, feature_reference):
    engineered_data = apply_feature_engineering(raw_data, feature_reference)
    X_input = prepare_feature_matrix(
        engineered_data,
        fitted_record["numeric_features"],
        fitted_record["categorical_features"],
    )
    return np.clip(fitted_record["model"].predict(X_input), 0, 5)


#### Run active model comparison

The constant mean baseline predicts the training-set average for every book. The median is theoretically optimal for MAE, but the mean provides the more intuitive "predict the average" reference point.

In [ ]:
# Evaluate the active configurations and the separate mean baseline on the fixed split.
model_results, fitted_models = evaluate_model_configs(active_model_configs)
constant_baseline_result = fit_and_evaluate_constant_baseline()
model_results = pd.concat(
    [model_results, pd.DataFrame([constant_baseline_result])],
    ignore_index=True,
)

model_results.sort_values("MAE").round({"MAE": 3, "RMSE": 3, "R2": 3})


### 3.8 Model Tuning

This section tunes one model at a time. The results from each tuning run are used to replace the default model configuration in `active_model_configs`.
For optimal results, we start by finding the best iterations/learning rate pairs for each model, then tune the remaining parameters.


#### Prepare the tuning grid

In [ ]:
RUN_TUNING = False

# Configure exactly one model family for each tuning run.
tuning_model_config = {
    "model": "CatBoost",
    "estimator": CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="MAE",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
    ),
    "param_grid": {
        "iterations": [400],
        "learning_rate": [0.09],
        "depth": [5, 6, 7],
        "l2_leaf_reg": [3.0, 5.0, 7.0],
    },
}

# HistGradientBoosting.
# tuning_model_config = {
#     "model": "HistGradientBoosting",
#     "estimator": HistGradientBoostingRegressor(
#         random_state=RANDOM_STATE,
#         early_stopping=False,
#     ),
#     "param_grid": {
#         "max_iter": [75],
#         "learning_rate": [0.13],
#         "max_leaf_nodes": [31, 63, 127],
#         "min_samples_leaf": [10, 20, 40],
#         "l2_regularization": [0.5, 1.0, 2.0],
#     },
# }

# XGBoost.
# tuning_model_config = {
#     "model": "XGBoost",
#     "estimator": XGBRegressor(
#         subsample=0.8,
#         colsample_bytree=0.8,
#         reg_lambda=5.0,
#         objective="reg:squarederror",
#         eval_metric="mae",
#         tree_method="hist",
#         random_state=RANDOM_STATE,
#         n_jobs=-1,
#     ),
#     "param_grid": {
#         "n_estimators": [400,600],
#         "learning_rate": [0.05,0.06],
#         "max_depth": [6,7],
#         "min_child_weight": [6,7],
#     },
# }

# Random Forest.
# tuning_model_config = {
#     "model": "Random Forest",
#     "estimator": RandomForestRegressor(
#         random_state=RANDOM_STATE,
#         n_jobs=-1,
#     ),
#     "param_grid": {
#         "n_estimators": [100, 200, 300],
#         "max_features": [0.6, 0.7, 0.8],
#         "min_samples_leaf": [1, 2, 3],
#     },
# }

# Extra Trees.
# tuning_model_config = {
#     "model": "Extra Trees",
#     "estimator": ExtraTreesRegressor(
#         random_state=RANDOM_STATE,
#         n_jobs=-1,
#     ),
#     "param_grid": {
#         "n_estimators": [100, 200, 300],
#         "max_features": [0.6, 0.7, 0.8],
#         "min_samples_leaf": [1, 2, 3],
#     },
# }


def expand_param_grid(param_grid):
    """Return every parameter combination from a grid dictionary."""
    parameter_names = list(param_grid.keys())
    parameter_values = [param_grid[name] for name in parameter_names]
    return [
        dict(zip(parameter_names, values))
        for values in product(*parameter_values)
    ]


if tuning_model_config["model"] not in MODEL_SPECS:
    raise ValueError(f"Unknown tuning model: {tuning_model_config['model']!r}")

tuning_parameter_sets = (
    expand_param_grid(tuning_model_config["param_grid"])
    if RUN_TUNING
    else []
)
tuning_grid_summary = pd.DataFrame([{
    "model": tuning_model_config["model"],
    "grid_configurations": len(tuning_parameter_sets),
    "status": "ready" if RUN_TUNING else "skipped",
}])

tuning_grid_summary

#### Run tuning when enabled

In [ ]:
# Build candidate configurations for the one selected model only.
if RUN_TUNING:
    tuning_model_configs = [
        {
            "model": tuning_model_config["model"],
            "estimator": clone(tuning_model_config["estimator"]).set_params(**params),
        }
        for params in tuning_parameter_sets
    ]
    tuning_results, fitted_tuned_models = evaluate_model_configs(tuning_model_configs)

    tuning_results_ranked = tuning_results.sort_values(
        by=["MAE", "RMSE", "R2"],
        ascending=[True, True, False],
    ).reset_index(drop=True)
    best_tuning_results = tuning_results_ranked.head(1).copy()
else:
    tuning_model_configs = []
    tuning_results = pd.DataFrame(columns=[
        "model",
        "n_numeric_features",
        "n_categorical_features",
        "parameters",
        "MAE",
        "RMSE",
        "R2",
    ])
    fitted_tuned_models = {}
    tuning_results_ranked = tuning_results.copy()
    best_tuning_results = tuning_results.copy()
    print(f"Skipping tuning for {tuning_model_config['model']} because RUN_TUNING is False.")

best_tuning_results.round({"MAE": 3, "RMSE": 3, "R2": 3})


#### Prepare tuning summary

We keep the best candidates for the selected model ready for display when tuning has been run, otherwise we show an empty result.


In [ ]:
# Prepare the top candidates for the selected model only.
if RUN_TUNING and not tuning_results_ranked.empty:
    top_tuning_results = tuning_results_ranked.head(20).reset_index(drop=True)
else:
    top_tuning_results = pd.DataFrame(columns=["model", "parameters", "MAE", "RMSE", "R2"])
    print("No tuning candidates to display because RUN_TUNING is False.")


#### Display tuning candidates

In [ ]:
top_tuning_results[["model", "parameters", "MAE", "RMSE", "R2"]].head(10)


### 3.9 Final Evaluation

The final comparison uses the active configurations from Section 3.7. Once manual tuning has found better parameters, we copy them into `active_model_configs` and rerun the normal comparison instead of depending on the tuning grid every time.


#### Rank final active models

In [ ]:
# Rank the active model configurations by MAE, with RMSE and R2 available for context.
final_model_results = (
    model_results.round({"MAE": 3, "RMSE": 3, "R2": 3})
    .sort_values(["MAE", "RMSE", "R2"], ascending=[True, True, False])
    .reset_index(drop=True)
)

final_model_results


#### Select the best fitted model

We store the best model from the active comparison for inspection and prediction checks.


In [ ]:
selected_model_name = None  # Set to a configured model name, or keep None for the best model.
# The baseline is comparison-only and can never become the selected/exported model.
selectable_model_results = final_model_results[
    final_model_results["model"].isin(MODEL_SPECS)
]
if selected_model_name is None:
    selected_model_name = selectable_model_results["model"].iloc[0]

if selected_model_name not in MODEL_SPECS:
    raise ValueError(
        f"Selected model {selected_model_name!r} is not a configured predictive model."
    )

selected_result_rows = selectable_model_results[
    selectable_model_results["model"] == selected_model_name
]
if len(selected_result_rows) != 1:
    raise ValueError(
        f"Expected one final result for {selected_model_name!r}; "
        f"found {len(selected_result_rows)}."
    )
best_final_model = selected_result_rows.iloc[0]

best_model_key = (best_final_model["model"], best_final_model["parameters"])
best_fitted_record = fitted_models[best_model_key]

print(
    f"Best final model: {best_final_model['model']} "
    f"with MAE = {best_final_model['MAE']:.3f}"
)

### 3.10 Feature Impact Analysis

This optional analysis answers two related questions:

1. How well does each feature perform by itself compared with the constant mean baseline?
2. How does test MAE change as features are added in that standalone ranking order?

Every score comes from a fresh clone of the selected model using its current estimator settings, preprocessing strategy, and fitted-record feature lists.

#### 3.10.1. Set analysis controls

Feature-impact analysis refits the selected model many times, so it can be switched off independently of the main model comparison.

In [ ]:
# Guard to avoid running the analysis multiple times.
RUN_FEATURE_IMPACT_ANALYSIS = True

# The final cumulative fit and a fresh full-feature fit should have the same MAE.
FEATURE_IMPACT_MAE_TOLERANCE = 1e-9

#### 3.10.2. Select the active model configuration

Use the configuration that produced the selected model in the active model comparison above.

In [ ]:
selected_model_spec = MODEL_SPECS[selected_model_name]

selected_model_configs = [
    config
    for config in active_model_configs
    if config["model"] == selected_model_name
]

if len(selected_model_configs) != 1:
    raise ValueError(
        f"Expected exactly one active configuration for {selected_model_name!r}; "
        f"found {len(selected_model_configs)}."
    )

selected_model_config = selected_model_configs[0]

#### 3.10.3. Read and validate the selected features

The fitted record is the source of truth because it contains the exact numeric and categorical feature lists evaluated above. The checks below catch stale notebook state before the expensive refits begin.

In [ ]:
selected_numeric_features = list(best_fitted_record["numeric_features"])
selected_categorical_features = list(best_fitted_record["categorical_features"])
selected_feature_order = selected_numeric_features + selected_categorical_features

if not selected_feature_order:
    raise ValueError(f"Selected model {selected_model_name!r} has no configured features.")

if len(selected_feature_order) != len(set(selected_feature_order)):
    raise ValueError(f"Selected model {selected_model_name!r} contains duplicate features.")

features_match_model_spec = (
    selected_numeric_features == list(selected_model_spec["numeric_features"])
    and selected_categorical_features == list(selected_model_spec["categorical_features"])
)
if not features_match_model_spec:
    raise ValueError(
        "The selected fitted record does not match the current MODEL_SPECS feature lists. "
        "Rerun the active model comparison before feature-impact analysis."
    )

#### 3.10.4. Prepare a feature subset

Validate the requested names, preserve the selected model's feature order, and build matching train and test matrices.

In [ ]:
selected_feature_set = set(selected_feature_order)


def prepare_feature_subset(feature_names):
    """Validate a feature subset and prepare its train and test matrices."""
    requested_features = list(feature_names)

    # Catch invalid requests before starting an expensive model fit.
    if not requested_features:
        raise ValueError("Feature-impact evaluation requires at least one feature.")
    if len(requested_features) != len(set(requested_features)):
        raise ValueError("Feature-impact subsets cannot contain duplicate feature names.")

    unknown_features = sorted(set(requested_features) - selected_feature_set)
    if unknown_features:
        raise ValueError(
            f"Features are not used by {selected_model_name!r}: {unknown_features}"
        )

    # Preserve the feature order used by the selected model.
    requested_feature_set = set(requested_features)
    numeric_features = [
        feature
        for feature in selected_numeric_features
        if feature in requested_feature_set
    ]
    categorical_features = [
        feature
        for feature in selected_categorical_features
        if feature in requested_feature_set
    ]

    X_train_subset = prepare_feature_matrix(
        X_train_engineered,
        numeric_features,
        categorical_features,
    )
    X_test_subset = prepare_feature_matrix(
        X_test_engineered,
        numeric_features,
        categorical_features,
    )

    return (
        X_train_subset,
        X_test_subset,
        numeric_features,
        categorical_features,
    )

#### 3.10.5. Fit and score a feature subset

Fit a fresh clone of the selected model with the same strategy used in the main model comparison, then calculate hold-out metrics.

In [ ]:
feature_impact_y_train = y_model.loc[MODEL_TRAIN_INDEX]
feature_impact_y_test = y_model.loc[MODEL_TEST_INDEX]


def fit_and_evaluate_feature_subset(feature_names):
    """Fit the selected model on a feature subset and return hold-out metrics."""
    (
        X_train_subset,
        X_test_subset,
        numeric_features,
        categorical_features,
    ) = prepare_feature_subset(feature_names)

    estimator = clone(selected_model_config["estimator"])
    fit_strategy = selected_model_spec["fit_strategy"]

    if fit_strategy == "sklearn_pipeline":
        fitted_model = build_sklearn_pipeline(
            estimator,
            numeric_features,
            categorical_features,
            scale_numeric=selected_model_spec["scale_numeric"],
        )
        fitted_model.fit(X_train_subset, feature_impact_y_train)
    elif fit_strategy == "native_categorical":
        fitted_model = estimator
        fitted_model.fit(
            X_train_subset,
            feature_impact_y_train,
            cat_features=categorical_features,
        )
    else:
        raise ValueError(
            f"Unknown fit strategy for {selected_model_name!r}: {fit_strategy!r}"
        )

    predictions = np.clip(fitted_model.predict(X_test_subset), 0, 5)
    return regression_metrics(feature_impact_y_test, predictions)

#### 3.10.6. Initialize the analysis

Reset the optional outputs so rerunning the notebook cannot leave results from an older model selection in memory.

In [ ]:
standalone_feature_results = None
cumulative_feature_results = None
feature_impact_consistency = None
feature_impact_waterfall = None

if RUN_FEATURE_IMPACT_ANALYSIS:
    baseline_mae = float(constant_baseline_result["MAE"])
    total_fits = 2 * len(selected_feature_order) + 1
    print(
        f"Running {total_fits} feature-impact fits for {selected_model_name} "
        "with its current active estimator parameters."
    )
else:
    print(
        "Feature-impact analysis is disabled. "
        "Set RUN_FEATURE_IMPACT_ANALYSIS = True to run it manually."
    )

#### 3.10.7. Rank features by standalone performance

Fit one model per feature. A positive `standalone_MAE_gain` means that the feature beats the constant mean baseline when used alone.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    standalone_rows = []
    feature_count = len(selected_feature_order)

    for position, feature_name in enumerate(selected_feature_order, start=1):
        print(f"Standalone {position}/{feature_count}: {feature_name}")

        metrics = fit_and_evaluate_feature_subset([feature_name])
        feature_type = (
            "numeric"
            if feature_name in selected_numeric_features
            else "categorical"
        )

        standalone_rows.append({
            "feature": feature_name,
            "feature_type": feature_type,
            "standalone_MAE": metrics["MAE"],
            "standalone_RMSE": metrics["RMSE"],
            "standalone_R2": metrics["R2"],
            "standalone_MAE_gain": baseline_mae - metrics["MAE"],
        })

    standalone_feature_results = (
        pd.DataFrame(standalone_rows)
        .sort_values(
            ["standalone_MAE_gain", "feature"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    result_features = standalone_feature_results["feature"].tolist()
    if len(result_features) != feature_count:
        raise AssertionError("Standalone results must contain every selected feature once.")
    if set(result_features) != set(selected_feature_order):
        raise AssertionError("Standalone results do not match the selected features.")

    ranked_features = result_features

##### Standalone results

Features are ordered from the largest to the smallest standalone MAE gain.

In [ ]:
standalone_feature_results.round(4) if RUN_FEATURE_IMPACT_ANALYSIS else None

##### Plot standalone MAE gain

Green bars improve on the constant mean baseline; orange bars perform worse than it.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    standalone_plot_data = standalone_feature_results.sort_values(
        "standalone_MAE_gain",
        ascending=True,
    )
    standalone_colors = [
        "#2a9d8f" if gain >= 0 else "#e76f51"
        for gain in standalone_plot_data["standalone_MAE_gain"]
    ]

    fig, ax = plt.subplots(
        figsize=(10, max(6, 0.4 * len(standalone_plot_data)))
    )
    ax.barh(
        standalone_plot_data["feature"],
        standalone_plot_data["standalone_MAE_gain"],
        color=standalone_colors,
    )
    ax.axvline(0, color="#495057", linewidth=1, linestyle="--")
    ax.set(
        xlabel="Standalone MAE gain versus constant mean baseline",
        ylabel="Feature",
        title=f"Standalone feature MAE gain — {selected_model_name}",
    )
    plt.tight_layout()
    plt.show()

#### 3.10.8. Add features cumulatively

Start with the strongest standalone feature, then add one feature at a time. `incremental_MAE_gain` measures the change caused by the latest addition, while `cumulative_MAE_gain` compares the current subset with the constant mean baseline.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    cumulative_rows = []
    cumulative_features = []
    previous_mae = baseline_mae
    feature_count = len(ranked_features)

    for step, feature_name in enumerate(ranked_features, start=1):
        cumulative_features.append(feature_name)
        print(f"Cumulative {step}/{feature_count}: adding {feature_name}")

        metrics = fit_and_evaluate_feature_subset(cumulative_features)
        current_mae = metrics["MAE"]

        cumulative_rows.append({
            "step": step,
            "feature_added": feature_name,
            #"features_used": tuple(cumulative_features),
            "n_features": len(cumulative_features),
            "MAE": current_mae,
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
            "incremental_MAE_gain": previous_mae - current_mae,
            "cumulative_MAE_gain": baseline_mae - current_mae,
        })

        previous_mae = current_mae

    cumulative_feature_results = pd.DataFrame(cumulative_rows)

##### Validate the cumulative sequence

The subset must grow by exactly one feature per step and finish with every selected feature.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    expected_feature_counts = list(range(1, len(selected_feature_order) + 1))
    actual_feature_counts = cumulative_feature_results["n_features"].tolist()

    if actual_feature_counts != expected_feature_counts:
        raise AssertionError("Cumulative feature counts must increase by exactly one.")
    if cumulative_features != ranked_features:
        raise AssertionError("Features were not added in standalone ranking order.")
    if set(cumulative_features) != set(selected_feature_order):
        raise AssertionError("The final subset does not contain every selected feature.")

##### Cumulative results

In [ ]:
cumulative_feature_results.round(4) if RUN_FEATURE_IMPACT_ANALYSIS else None

#### 3.10.9. Check the full-feature result

Refit the complete selected-model configuration once more. Its MAE should match the last cumulative step within the stated numerical tolerance.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    print("Consistency fit: refitting the complete selected-model configuration.")

    full_subset_metrics = fit_and_evaluate_feature_subset(selected_feature_order)
    final_cumulative_mae = float(cumulative_feature_results.iloc[-1]["MAE"])
    fresh_full_mae = float(full_subset_metrics["MAE"])
    absolute_mae_difference = abs(final_cumulative_mae - fresh_full_mae)
    consistency_check_passed = bool(np.isclose(
        final_cumulative_mae,
        fresh_full_mae,
        rtol=FEATURE_IMPACT_MAE_TOLERANCE,
        atol=FEATURE_IMPACT_MAE_TOLERANCE,
    ))

    feature_impact_consistency = pd.DataFrame([{
        "model": selected_model_name,
        "features_match_complete_configuration": (
            cumulative_features == ranked_features
            and set(cumulative_features) == set(selected_feature_order)
        ),
        "final_cumulative_MAE": final_cumulative_mae,
        "fresh_full_configuration_MAE": fresh_full_mae,
        "absolute_MAE_difference": absolute_mae_difference,
        "consistency_check_passed": consistency_check_passed,
    }])

    if not consistency_check_passed:
        raise AssertionError(
            "Final cumulative MAE does not match a fresh full-configuration fit."
        )

##### Consistency-check result

In [ ]:
feature_impact_consistency.round(10) if RUN_FEATURE_IMPACT_ANALYSIS else None

#### 3.10.10. Prepare the MAE waterfall

Convert the cumulative MAE path into step-by-step changes for the final chart.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    cumulative_mae_values = cumulative_feature_results["MAE"].tolist()
    mae_path = [baseline_mae] + cumulative_mae_values
    waterfall_changes = [
        current_mae - previous_mae
        for previous_mae, current_mae in zip(mae_path[:-1], mae_path[1:])
    ]

    waterfall_labels = (
        ["Constant mean baseline"]
        + ranked_features
        + [f"Full {selected_model_name}"]
    )

    feature_hover_text = []
    for feature_name, mae_change, current_mae in zip(
        ranked_features,
        waterfall_changes,
        cumulative_mae_values,
    ):
        feature_hover_text.append(
            f"<b>{feature_name}</b><br>MAE change: {mae_change:+.6f}"
            f"<br>Test MAE after step: {current_mae:.6f}"
        )

    waterfall_hover_text = (
        [f"<b>Constant mean baseline</b><br>Test MAE: {baseline_mae:.6f}"]
        + feature_hover_text
        + [
            f"<b>Full {selected_model_name}</b>"
            f"<br>Test MAE: {final_cumulative_mae:.6f}"
        ]
    )

#### 3.10.11. Plot the MAE waterfall

A downward step improves MAE; an upward step makes MAE worse at that point in the ranked sequence.

In [ ]:
if RUN_FEATURE_IMPACT_ANALYSIS:
    waterfall_measures = (
        ["absolute"]
        + ["relative"] * len(ranked_features)
        + ["total"]
    )
    waterfall_values = [baseline_mae] + waterfall_changes + [0]
    waterfall_text = (
        [f"{baseline_mae:.4f}"]
        + [f"{change:+.4f}" for change in waterfall_changes]
        + [f"{final_cumulative_mae:.4f}"]
    )

    feature_impact_waterfall = go.Figure(go.Waterfall(
        name="Test MAE",
        orientation="v",
        measure=waterfall_measures,
        x=waterfall_labels,
        y=waterfall_values,
        text=waterfall_text,
        textposition="outside",
        hovertext=waterfall_hover_text,
        hoverinfo="text",
        decreasing={"marker": {"color": "#2a9d8f"}},
        increasing={"marker": {"color": "#e76f51"}},
        totals={"marker": {"color": "#457b9d"}},
        connector={"line": {"color": "#7f8c8d"}},
    ))
    feature_impact_waterfall.update_layout(
        title=(
            "Cumulative MAE improvement from ranked feature addition — "
            f"{selected_model_name}"
        ),
        yaxis_title="Test MAE",
        xaxis_title="Ranked feature addition",
        showlegend=False,
        height=650,
        margin={"b": 180},
    )
    feature_impact_waterfall.update_xaxes(tickangle=-45)
    feature_impact_waterfall.show()

#### Interpretation

- **Standalone MAE gain** measures how useful a feature is by itself compared with the constant mean baseline.
- **Incremental MAE gain** measures what a feature adds after all higher-ranked features are already included.
- A feature can perform well alone but add little later when related features already contain similar information.
- Results can change with `selected_model_name` because model families use information differently.
- The cumulative path depends on the standalone ranking order.

This analysis measures predictive usefulness, not causal impact.

### 3.11 Hold-out prediction diagnostics

The plots below use the untouched test set. The dashed diagonal represents perfect predictions; the error distribution should be centred near zero without a strong skew.


#### Compare actual and predicted ratings

We inspect both individual test-set predictions and their errors to identify systematic over- or under-prediction.


In [ ]:
# Diagnose the selected model using the same held-out rows used for final evaluation.
y_test = y_model.loc[MODEL_TEST_INDEX]
best_test_predictions = predict_from_raw_input(
    best_fitted_record,
    X_model_raw.loc[MODEL_TEST_INDEX],
    feature_engineering_reference,
)

evaluation_details = pd.DataFrame({
    "actual_rating": y_test,
    "predicted_rating": best_test_predictions,
})
evaluation_details["prediction_error"] = (evaluation_details["predicted_rating"] - evaluation_details["actual_rating"])
evaluation_details["absolute_error"] = evaluation_details["prediction_error"].abs()

rating_min = min(evaluation_details["actual_rating"].min(),evaluation_details["predicted_rating"].min(),)
rating_max = max(evaluation_details["actual_rating"].max(),evaluation_details["predicted_rating"].max(),)
axis_padding = max((rating_max - rating_min) * 0.05, 0.02)
axis_min = max(0, rating_min - axis_padding)
axis_max = min(5, rating_max + axis_padding)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=evaluation_details,
    x="actual_rating",
    y="predicted_rating",
    alpha=0.35,
    color="#2a9d8f",
    ax=axes[0],
)
axes[0].plot([axis_min, axis_max], [axis_min, axis_max], "--", color="#e76f51", label="Perfect prediction")
axes[0].set(
    xlim=(axis_min, axis_max),
    ylim=(axis_min, axis_max),
    xlabel="Actual average rating",
    ylabel="Predicted average rating",
    title=f"Actual vs predicted ratings : {selected_model_name}",
)
axes[0].legend()

sns.histplot(
    data=evaluation_details,
    x="prediction_error",
    bins=30,
    kde=True,
    color="#457b9d",
    ax=axes[1],
)
axes[1].axvline(0, linestyle="--", color="#e76f51", label="No error")
axes[1].set(
    xlabel="Prediction error (predicted - actual)",
    ylabel="Books",
    title="Distribution of prediction errors",
)
axes[1].legend()

plt.tight_layout()
plt.show()

evaluation_details.head(10)


#### Interpretation

Linear regression is expected to be weaker because it mainly learns additive linear effects after scaling and one-hot encoding. Goodreads ratings are likely affected by non-linear relationships between popularity, publication timing, page count, review activity, author metadata, and publisher metadata.

`HistGradientBoostingRegressor` and `XGBRegressor` capture non-linear patterns with boosted trees. `RandomForestRegressor` and `ExtraTreesRegressor` provide bagged and randomized tree alternatives using the same compact encoded features. `CatBoostRegressor` can additionally use high-cardinality categorical metadata such as `first_author` and `publisher` directly.

------------
## 4. Model export

This section refit the selected model on all eligible rows and serialize one self-contained Python predictor using `cloudpickle` for the web application.

### 4.1 Export controls

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import platform

import catboost as catboost_package
import cloudpickle
import sklearn as sklearn_package
import xgboost as xgboost_package

EXPORT_MODEL = True
OVERWRITE_MODEL = True
MODEL_TO_EXPORT = best_final_model["model"]
MODEL_EXPORT_PATH = Path(f"../models/goodreads_rating_predictor_{MODEL_TO_EXPORT}.pkl")

export_controls = pd.DataFrame([
    {"setting": "export_enabled", "value": EXPORT_MODEL},
    {"setting": "overwrite_enabled", "value": OVERWRITE_MODEL},
    {"setting": "model_to_export", "value": MODEL_TO_EXPORT},
    {"setting": "artifact_path", "value": str(MODEL_EXPORT_PATH)},
])
export_controls


### 4.2 Exportable predictor

The predictor stores the fitted estimator, its train-fitted feature references, the selected feature lists, and the two notebook functions needed to turn raw rows into model-ready inputs.

In [ ]:
class GoodreadsRatingPredictor:
    """Self-contained raw-input predictor exported from this notebook."""

    def __init__(
        self,
        fitted_record,
        feature_reference,
        metadata,
        feature_engineer=apply_feature_engineering,
        matrix_preparer=prepare_feature_matrix,
    ):
        self.model = fitted_record["model"]
        self.feature_reference = feature_reference
        self.numeric_features = list(fitted_record["numeric_features"])
        self.categorical_features = list(fitted_record["categorical_features"])
        self.metadata = metadata
        self._feature_engineer = feature_engineer
        self._matrix_preparer = matrix_preparer

    def prepare_features(self, raw_data: pd.DataFrame) -> pd.DataFrame:
        """Transform raw book rows into the columns expected by the fitted model."""
        engineered = self._feature_engineer(raw_data, self.feature_reference)
        return self._matrix_preparer(
            engineered,
            self.numeric_features,
            self.categorical_features,
        )

    def predict(self, raw_data: pd.DataFrame) -> np.ndarray:
        """Predict ratings from raw rows and keep predictions on the 0-to-5 scale."""
        model_input = self.prepare_features(raw_data)
        return np.clip(self.model.predict(model_input), 0, 5)


### 4.3 Refit and export

In [ ]:
exported_predictor = None

if not EXPORT_MODEL:
    print("Model export is disabled. Set EXPORT_MODEL = True to refit and export.")
else:
    matching_configs = [
        config for config in active_model_configs
        if config["model"] == MODEL_TO_EXPORT
    ]
    if len(matching_configs) != 1:
        raise ValueError(
            f"Expected exactly one active configuration for {MODEL_TO_EXPORT!r}; "
            f"found {len(matching_configs)}."
        )

    if MODEL_EXPORT_PATH.exists() and not OVERWRITE_MODEL:
        raise FileExistsError(
            f"Artifact already exists: {MODEL_EXPORT_PATH.resolve()}. "
            "Set OVERWRITE_MODEL = True to replace it."
        )

    export_config = matching_configs[0]
    export_feature_reference = fit_feature_engineering_reference(X_model_raw, y_model)
    export_engineered_data = apply_feature_engineering(
        X_model_raw,
        export_feature_reference,
    )
    export_fitted_record = fit_model_configuration(
        export_config,
        export_engineered_data,
        y_model,
    )

    export_metadata = {
        "artifact_format_version": 1,
        "exported_at_utc": datetime.now(timezone.utc).isoformat(),
        "model_name": MODEL_TO_EXPORT,
        "parameters": export_fitted_record["parameters"],
        "training_rows": len(X_model_raw),
        "target": TARGET,
        "raw_input_columns": list(RAW_MODEL_INPUT_COLUMNS),
        "numeric_features": export_fitted_record["numeric_features"],
        "categorical_features": export_fitted_record["categorical_features"],
        "minimum_ratings_count_for_training": MIN_RATINGS_COUNT_FOR_MODELLING,
        "reference_year": REFERENCE_YEAR,
        "library_versions": {
            "python": platform.python_version(),
            "pandas": pd.__version__,
            "numpy": np.__version__,
            "scikit_learn": sklearn_package.__version__,
            "catboost": catboost_package.__version__,
            "xgboost": xgboost_package.__version__,
            "cloudpickle": cloudpickle.__version__,
        },
    }

    exported_predictor = GoodreadsRatingPredictor(
        fitted_record=export_fitted_record,
        feature_reference=export_feature_reference,
        metadata=export_metadata,
    )

    MODEL_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with MODEL_EXPORT_PATH.open("wb") as artifact_file:
        cloudpickle.dump(exported_predictor, artifact_file)

    print(f"Exported {MODEL_TO_EXPORT} to {MODEL_EXPORT_PATH.resolve()}")


### 4.4 Reload verification

The saved predictor is reloaded and compared on ordinary rows plus one deliberately imperfect row containing missing, invalid, unseen, and extra values.

In [ ]:
if not EXPORT_MODEL:
    print("Reload verification is skipped because export is disabled.")
else:
    with MODEL_EXPORT_PATH.open("rb") as artifact_file:
        reloaded_predictor = cloudpickle.load(artifact_file)

    verification_input = X_model_raw.head(10).copy().reset_index(drop=True)
    imperfect_row = verification_input.head(1).astype("object").copy()
    imperfect_row.loc[0, "authors"] = "Previously Unseen Author"
    imperfect_row.loc[0, "publisher"] = None
    imperfect_row.loc[0, "language_code"] = "new_language"
    imperfect_row.loc[0, "num_pages"] = np.nan
    imperfect_row.loc[0, "ratings_count"] = "250"
    imperfect_row.loc[0, "text_reviews_count"] = "not_available"
    imperfect_row.loc[0, "publication_date"] = "invalid_date"
    imperfect_row["extra_unused_column"] = "ignored safely"
    verification_input = pd.concat([verification_input, imperfect_row], ignore_index=True)

    predictions_before_reload = exported_predictor.predict(verification_input)
    predictions_after_reload = reloaded_predictor.predict(verification_input)
    np.testing.assert_allclose(predictions_before_reload, predictions_after_reload)
    assert reloaded_predictor.metadata == exported_predictor.metadata

    export_verification_summary = pd.DataFrame([{
        "artifact": str(MODEL_EXPORT_PATH.resolve()),
        "file_size_bytes": MODEL_EXPORT_PATH.stat().st_size,
        "verified_rows": len(verification_input),
        "predictions_identical": True,
    }])


In [ ]:
export_verification_summary